[目录](./table_of_contents.ipynb)

# 设计卡尔曼滤波器

In [ ]:
%matplotlib inline

In [ ]:
#格式化书本
import book_format
book_format.set_style()

## 介绍

上一章中，我们解决了一些“教科书式”的问题。这些问题很容易陈述，只需几行代码即可编程，并且易于教学。现实世界中的问题很少如此简单。在这一章中，我们将处理更为现实的示例，并学习如何评估滤波器的性能。

我们将从跟踪二维空间中的机器人开始，例如田野或仓库。我们将从一个简单的噪声传感器开始，该传感器输出嘈杂的 $(x,y)$ 坐标，我们需要对其进行滤波以生成2D轨迹。一旦我们掌握了这个概念，我们将通过增加更多传感器并添加控制输入来显著扩展问题。

我们接下来会处理一个非线性问题。世界是非线性的，但是卡尔曼滤波器是线性的。有时你可以在轻度非线性问题中使用它，有时则不行。我将向您展示两种情况的示例。这将为本书的剩余部分设定舞台，我们将在其中学习非线性问题的技术。

## 跟踪机器人

这个跟踪机器人的首次尝试将与前几章的一维狗跟踪问题非常相似。我们现在有一个传感器，它在二维空间中提供传感器位置的嘈杂测量，而不是在走廊中输出位置。在每个时间 $t$，它将提供传感器在场地中位置的 $(x,y)$ 坐标对的嘈杂测量。

此书的实现与真实传感器的交互的代码不在本书的范围之内，因此我们将编写传感器的简单模拟。我们将在编写过程中开发多个具有不同复杂性的传感器，因此在编写它们时，我将在函数名后附加一个数字。

因此，让我们从一个非常简单的传感器开始，模拟跟踪直线行驶的物体。它通过初始位置、速度和噪声标准差进行初始化。每次调用 `read()` 都会将位置更新一个时间步长并返回新的测量值。

In [ ]:
from numpy.random import randn

In [ ]:
class PosSensor(object):
    def __init__(self, pos=(0, 0), vel=(0, 0), noise_std=1.):
        self.vel = vel
        self.noise_std = noise_std
        self.pos = [pos[0], pos[1]]
        
    def read(self):
        self.pos[0] += self.vel[0]
        self.pos[1] += self.vel[1]
        
        return [self.pos[0] + randn() * self.noise_std,
                self.pos[1] + randn() * self.noise_std]

快速测试以验证它是否按我们的预期工作。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from kf_book.book_plots import plot_measurements

pos, vel = (4, 3), (2, 1)
sensor = PosSensor(pos, vel, noise_std=1)
ps = np.array([sensor.read() for _ in range(50)])
plot_measurements(ps[:, 0], ps[:, 1]);

看起来没问题。斜率为1/2，符合速度为(2,1)的预期，数据似乎开始于接近(6,4)。但这看起来不够现实。这仍然是一个“教科书式”的表示。随着我们的继续，我们将增加一些复杂性，以增加真实世界的行为。

### 选择状态变量

像往常一样，第一步是选择我们的状态变量。我们在两个维度上进行跟踪，有一个传感器可以给我们每个维度的读数，因此我们知道我们有两个“观察变量”$x$和$y$。如果我们仅使用这两个变量创建卡尔曼滤波器，则性能将不会很好，因为我们将忽略速度可以提供给我们的信息。我们还将希望将速度纳入我们的方程式中。我将把它表示为

$$\mathbf x = 
\begin{bmatrix}x & \dot x & y & \dot y\end{bmatrix}^\mathsf T$$

这个组织没有什么特别的地方。我本来可以使用 $\begin{bmatrix}x & y & \dot x &  \dot y\end{bmatrix}^\mathsf T$ 或者其他不那么合乎逻辑的东西。我只需要在其它矩阵中保持一致。我喜欢将位置和速度放在一起，因为这样可以将位置和速度之间的协方差保持在协方差矩阵的同一子块中。在我的公式中，`P[1,0]` 包含了 $x$ 和 $\dot x$ 的协方差。在另一种公式中，该协方差在 `P[2,0]`。随着维数的增加，情况会变得更糟。

让我们停下来讨论一下如何识别隐藏的变量。这个例子有点显而易见，因为我们已经解决了一维情况，但其他问题可能不太明显。这个问题没有简单的答案。首先要问自己的是，传感器数据的一阶和二阶导数的解释是什么。我们这样做是因为，如果您使用固定的时间步长从传感器中读取数据，则获得一阶和二阶导数在数学上是微不足道的。第一导数就是两个连续读数之间的差异。在我们的跟踪案例中，第一导数有一个明显的物理解释：两个连续位置之间的差异是速度。

除此之外，您可以开始思考如何将两个或多个不同传感器的数据组合起来以产生更多信息。这就打开了传感器融合领域，我们将在后面的部分中介绍一些例子。现在，认识到选择适当的状态变量对于从您的滤波器获得最佳性能至关重要。一旦您选择了隐藏变量，您必须运行多个测试以确保您正在为其生成真实结果。Kalman滤波器运行您提供的任何模型；如果您的模型无法为隐藏变量生成良好的信息，则Kalman滤波器输出将是荒谬的。

### 设计状态转移函数

我们的下一步是设计状态转移函数。回想一下，状态转移函数被实现为一个矩阵 $\mathbf F$，我们将其乘以我们系统的前一个状态以获得下一个状态，就像这样。

$$\mathbf{\bar x} = \mathbf{Fx}$$

我不会过多阐述，因为它与我们在上一章中所做的一维案例非常相似。状态转移方程是

$$
\begin{aligned}
x &= 1x + \Delta t \dot x + 0y + 0 \dot y \\
v_x &= 0x + 1\dot x + 0y + 0 \dot y \\
y &= 0x + 0\dot x + 1y + \Delta t \dot y \\
v_y &= 0x + 0\dot x + 0y + 1 \dot y
\end{aligned}
$$

这样排列可以向我们展示出$\small\mathbf F$所需的值和行列组织。我们将其转换为矩阵向量形式：

$$
\begin{bmatrix}x \\ \dot x \\ y \\ \dot y\end{bmatrix} = \begin{bmatrix}1& \Delta t& 0& 0\\0& 1& 0& 0\\0& 0& 1& \Delta t\\ 0& 0& 0& 1\end{bmatrix}\begin{bmatrix}x \\ \dot x \\ y \\ \dot y\end{bmatrix}$$

所以，让我们在Python中实现这个。这很简单；这里唯一的新内容是将`dim_z`设置为2。我们将在第4步中了解它为什么设置为2。

In [ ]:
from filterpy.kalman import KalmanFilter

tracker = KalmanFilter(dim_x=4, dim_z=2)
dt = 1.   # time step 1 second

In [ ]:
tracker.F = np.array([[1, dt, 0, 0],
                      [0, 1, 0, 0],
                      [0, 0, 1, dt],
                      [0, 0, 0, 1]])
                      
### 设计过程噪声矩阵

FilterPy 可以为我们计算 $\mathbf Q$ 矩阵。为简单起见，我将假设noise是离散时间维纳过程——在每个时间段内是恒定的。这个假设允许我使用方差来指定我认为模型在步骤之间改变了多少。如果不清楚，请重新查看KalmanFilter数学章节。

```python
from scipy.linalg import block_diag
from filterpy.common import Q_discrete_white_noise

q = Q_discrete_white_noise(dim=2, dt=dt, var=0.001)
tracker.Q = block_diag(q, q)
print(tracker.Q)

这里我假设 x 和 y 上的噪声是独立的，因此任何 x 和 y 变量之间的协方差应该为零。这使我能够为一个维度计算 $\mathbf Q$，然后使用 `block_diag` 将其复制到 x 和 y 轴。

### 设计控制函数

In [ ]:

我们还没有为机器人添加控制，因此在此步骤中没有什么可做的。 `KalmanFilter` 类在假定没有控制输入的情况下将 `B` 初始化为零，因此无需编写代码。如果您愿意，可以明确设置 `tracker.B` 为0，但是您可以看到它已经具有该值。

```python
tracker.B

### 设计测量函数

测量函数 $\mathbf H$ 定义了我们如何使用方程 $\mathbf z = \mathbf{Hx}$ 从状态变量到测量值。在这种情况下，我们对 (x，y) 进行测量，因此我们将把 $\mathbf z$ 设计为 $\begin{bmatrix}x & y\end{bmatrix}^\mathsf T$，其维度为 2x1。我们的状态变量的大小为 4x1。我们可以通过回忆矩阵大小为 MxN 乘以 NxP 会产生大小为 MxP 的矩阵来推断出 $\textbf{H}$ 的所需大小。因此，

$$(2\times 1) = (a\times b)(4 \times 1) = (2\times 4)(4\times 1)$$

因此，$\textbf{H}$ 为 2x4。

填写$\textbf{H}$的值很容易，因为测量值是机器人的位置，即状态$\textbf{x}$的$x$和$y$变量。让我们通过决定更改单位使其稍微有趣一些。测量值以英尺为单位返回，而我们希望使用米为单位工作。$\textbf{H}$从状态到测量值的变换是$\mathsf{feet}=\mathsf{meters}/0.3048$。这样得到

$$\mathbf H =
\begin{bmatrix} 
\frac{1}{0.3048} & 0 & 0 & 0 \\
0 & 0 & \frac{1}{0.3048} & 0
\end{bmatrix}
$$

它对应于下列线性方程组

$$
\begin{aligned}
z_x &= (\frac{x}{0.3048}) + (0* v_x) + (0*y) + (0 * v_y) = \frac{x}{0.3048}\\
z_y &= (0*x) + (0* v_x) + (\frac{y}{0.3048}) + (0 * v_y) = \frac{y}{0.3048}
\end{aligned}
$$

这是一个简单的问题，我们可以直接找到方程，而不需要像我上面做的那样通过尺寸分析。但是，记住卡尔曼滤波器的方程对于所有矩阵都暗示了特定的尺寸，当我开始在设计某些东西时变得困惑时，查看矩阵维度是有用的。

这是我的实现：

In [ ]:
tracker.H = np.array([[1/0.3048, 0, 0,        0],
                      [0,        0, 1/0.3048, 0]])

### 设计测量噪声矩阵

我们假设 $x$ 和 $y$ 变量是独立的白色高斯过程。也就是说，$x$ 中的噪声与 $y$ 中的噪声不以任何方式相互依赖，并且噪声通常围绕平均值0分布。现在，让我们将 $x$ 和 $y$ 的方差设为5米$^2$。它们是独立的，因此没有协方差，我们的非对角线为0。这给了我们：

$$\mathbf R = \begin{bmatrix}\sigma_x^2 & \sigma_y\sigma_x \\ \sigma_x\sigma_y & \sigma_{y}^2\end{bmatrix} 
= \begin{bmatrix}5&0\\0&5\end{bmatrix}$$

这是一个 $2{\times}2$ 矩阵，因为我们有 2 个传感器输入，协方差矩阵总是 $n$ 个变量的 $n{\times}n$ 大小。在 Python 中，我们写成：

In [ ]:
tracker.R = np.array([[5., 0],
                      [0, 5]])
tracker.R

### 初始条件

对于我们的简单问题，我们将初始位置设置为 (0,0)，速度为 (0,0)。因为这是一个纯猜测，我们将协方差矩阵 $\small\mathbf P$ 设为一个大值。

$$ \mathbf x = \begin{bmatrix}0\\0\\0\\0\end{bmatrix}, \,
\mathbf P = \begin{bmatrix}500&0&0&0\\0&500&0&0\\0&0&500&0\\0&0&0&500\end{bmatrix}$$

Python 实现如下：

In [ ]:
tracker.x = np.array([[0, 0, 0, 0]]).T
tracker.P = np.eye(4) * 500.

### 实现滤波器

设计完成，现在我们只需编写代码来运行滤波器并以我们选择的格式输出数据。我们将运行 30 次代码。

In [ ]:
from filterpy.stats import plot_covariance_ellipse
from kf_book.book_plots import plot_filter

R_std = 0.35
Q_std = 0.04

def tracker1():
    tracker = KalmanFilter(dim_x=4, dim_z=2)
    dt = 1.0   # 时间步长

    tracker.F = np.array([[1, dt, 0,  0],
                          [0,  1, 0,  0],
                          [0,  0, 1, dt],
                          [0,  0, 0,  1]])
    tracker.u = 0.
    tracker.H = np.array([[1/0.3048, 0, 0, 0],
                          [0, 0, 1/0.3048, 0]])

    tracker.R = np.eye(2) * R_std**2
    q = Q_discrete_white_noise(dim=2, dt=dt, var=Q_std**2)
    tracker.Q = block_diag(q, q)
    tracker.x = np.array([[0, 0, 0, 0]]).T
    tracker.P = np.eye(4) * 500.
    return tracker

# 模拟机器人运动
N = 30
sensor = PosSensor((0, 0), (2, .2), noise_std=R_std)

zs = np.array([sensor.read() for _ in range(N)])

# 运行滤波器
robot_tracker = tracker1()
mu, cov, _, _ = robot_tracker.batch_filter(zs)

for x, P in zip(mu, cov):
    # x 和 y 的协方差
    cov = np.array([[P[0, 0], P[2, 0]], 
                    [P[0, 2], P[2, 2]]])
    mean = (x[0, 0], x[2, 0])
    plot_covariance_ellipse(mean, cov=cov, fc='g', std=3, alpha=0.5)
    
# 绘制结果
zs *= .3048 # 转换为米
plot_filter(mu[:, 0], mu[:, 2])
plot_measurements(zs[:, 0], zs[:, 1])
plt.legend(loc=2)
plt.xlim(0, 20);

In [ ]:
我鼓励你尝试将 $\mathbf Q$ 和 $\mathbf R$ 设置为不同的值。然而，在上一章中，我们已经进行了相当多的这种尝试，我们还有很多材料需要涵盖，因此我将继续前往更复杂的情况，我们也将有机会体验更改这些值。

我用绿色绘制了$x$和$y$的$3\sigma$协方差椭圆。可以解释一下它们的形状吗？也许你期望看到一个倾斜的椭圆，就像前面的章节中一样。如果是这样，请记住，在那些章节中，我们没有绘制$x$相对于$y$的图像，而是绘制$x$相对于$\dot x$的图像。$x$与$\dot x$相关，但$x$与$y$不相关或不依赖。因此，我们的椭圆不会倾斜。此外，$x$和$y$的noise都被建模为具有相同的noise标准差。如果我们将$R$设置为，

$$\mathbf R = \begin{bmatrix}1&0\\0&.5\end{bmatrix}$$

我们将告诉Kalman滤波器$x$中的noise比$y$中的noise更多，我们的椭圆将比它们高。

最终的 $\mathbf P$ 值告诉我们关于状态变量之间的相关性所需的全部信息。如果仅查看对角线，我们会看到每个变量的方差。换句话说，$\mathbf P_{0,0}$ 是 $x$ 的方差，$\mathbf P_{1,1}$ 是 $\dot x$ 的方差，$\mathbf P_{2,2}$ 是 $y$ 的方差，$\mathbf P_{3,3}$ 是 $\dot y$ 的方差。我们可以使用 `numpy.diag()` 提取矩阵的对角线。

```python
print(np.diag(robot_tracker.P))

协方差矩阵包含四个 $2{\times}2$ 矩阵，您应该能够轻松地挑选出来。这是由于 $x$ 与 $\dot x$ 的相关性以及 $y$ 与 $\dot y$ 的相关性。左上角显示了 $x$ 与 $\dot x$ 的协方差。

In [ ]:
c = robot_tracker.P[0:2, 0:2]
print(c)
plot_covariance_ellipse((0, 0), cov=c, fc='g', alpha=0.2)

协方差矩阵在左上角包含了$x$和$\dot x$的数据，因为它的组织方式。记住，$\mathbf P_{i,j}$和$\mathbf P_{j,i}$的条目包含$\sigma_i\sigma_j$。

最后，让我们看看$\mathbf P$的左下角，其中全是0。为什么是0？考虑$\mathbf P_{3,0}$。它存储了$\sigma_3\sigma_0$这个术语，即$\dot y$和$x$之间的协方差。这些是独立的，所以这个术语将为0。其余的术语是针对类似独立变量的情况。

In [ ]:
robot_tracker.P[2:4, 0:2]

## 滤波器顺序

我们只研究了跟踪位置和速度。这很有效，但这仅适用于我已经选择了适当的问题。现在你已经有了足够的卡尔曼滤波器经验，可以更加一般地考虑这个问题。

什么是”order”？在这些系统模型的背景下，它是准确建模系统所需的导数数量。考虑一个不会改变的系统，例如建筑物的高度。没有改变，因此不需要导数，系统的阶数为零。我们可以用方程式$x = 312.5$来表示这一点。

一阶系统具有一阶导数。例如，位置的变化是速度，我们可以写成

$$ v = \frac{dx}{dt}$$

将其积分成牛顿方程

$$ x = vt + x_0.$$

这也称为*恒速模型*，因为假设了恒定速度。

二阶系统具有二阶导数。位置的二阶导数是加速度，它的方程式是

$$a = \frac{d^2x}{dt^2}$$

将其积分成

$$ x = \frac{1}{2}at^2 +v_0t + x_0.$$

这也被称为*恒定加速度*模型。

另一个等价的看待方式是考虑多项式的阶数。恒定加速度模型具有二阶导数，因此它是二阶的。同样，多项式 $x = \frac{1}{2}at^2 +v_0t + x_0$ 是二阶的。

当我们设计状态变量和过程模型时，我们必须选择我们想要建模的系统的阶数。假设我们正在跟踪具有恒定速度的某物体。没有一个真实的世界过程是完美的，因此速度在短时间内会有轻微的变化。您可能会推断最好的方法是使用二阶滤波器，使加速度项处理速度的轻微变化。

实际上，那样做效果不佳。为了彻底理解这个问题，让我们看看使用与被滤波系统阶数不匹配的过程模型的效果。

首先，我们需要一个要滤波的系统。我将编写一个类来模拟具有恒定速度的对象。实际上，没有真正具有恒定速度的物理系统，因此在每次更新时，我们会通过微小的变化来更改速度。我还编写了一个传感器来模拟传感器中的高斯噪声。以下是代码，并且我绘制了一个示例运行以验证它是否正确工作。

In [ ]:
from kf_book.book_plots import plot_track

class ConstantVelocityObject(object):
    def __init__(self, x0=0, vel=1., noise_scale=0.06):
        self.x = x0
        self.vel = vel
        self.noise_scale = noise_scale

    def update(self):
        self.vel += randn() * self.noise_scale
        self.x += self.vel
        return (self.x, self.vel)

def sense(x, noise_scale=1.):
    return x[0] + randn()*noise_scale

np.random.seed(124)
obj = ConstantVelocityObject()

xs, zs = [], []
for i in range(50):
    x = obj.update()
    z = sense(x)
    xs.append(x)
    zs.append(z)

xs = np.asarray(xs)

In [ ]:
plot_track(xs[:, 0])
plot_measurements(range(len(zs)), zs)
plt.legend(loc='best');

我对这个图很满意。由于我们加入了系统噪声，轨迹不是完全直线的 - 这可能是一个人走在街上的轨迹，或者是一架受到变化多端的风的影响的飞机的轨迹。这里没有故意的加速度，因此我们称之为恒定速度系统。你可能会问自己，既然实际上存在微小的加速度，为什么我们不使用二阶卡尔曼滤波器来解决这些变化呢？让我们来找出答案。

如何设计零阶、一阶或二阶卡尔曼滤波器？我们一直在做这个，只是没有使用这些术语。可能有点繁琐，但我将详细阐述每一个 - 如果你已经明白了这个概念，可以随意略过一些内容。

### 零阶卡尔曼滤波器

一个零阶卡尔曼滤波器只是一个没有导数的滤波器。我们正在跟踪位置，这意味着我们只有一个位置状态变量（没有速度或加速度），状态转移函数也只考虑位置。使用矩阵形式，我们会说状态变量为

$$\mathbf x = \begin{bmatrix}x\end{bmatrix}$$

状态转移函数非常简单。位置没有变化，因此我们需要建模 $x=x$; 换句话说，时间 t+1 时的 *x* 与时间 t 时相同。用矩阵形式表示，我们的状态转移函数为

$$\mathbf F = \begin{bmatrix}1\end{bmatrix}$$

测量函数非常简单。回想一下，我们需要定义如何将状态变量 $\mathbf x$ 转换为测量值。我们将假设我们的测量值是位置。状态变量仅包含位置，因此我们有

$$\mathbf H = \begin{bmatrix}1\end{bmatrix}$$

让我们编写一个函数来构建并返回一个零阶卡尔曼滤波器。

In [ ]:
def ZeroOrderKF(R, Q, P=20):
    """ 创建零阶KalmanFilter。
    将 R 和 Q 指定为浮点数。"""
    
    kf = KalmanFilter(dim_x=1, dim_z=1)
    kf.x = np.array([0.])
    kf.R *= R
    kf.Q *= Q
    kf.P *= P
    kf.F = np.eye(1)
    kf.H = np.eye(1)
    return kf

### 一阶卡尔曼滤波器

一阶卡尔曼滤波器跟踪一阶系统，例如位置和速度。我们已经在上面的狗跟踪问题中完成了这个，所以这应该非常清楚。但让我们再次做一遍。

一阶系统有位置和速度，因此状态变量需要这两个变量。矩阵公式可以写成：

$$ \mathbf x = \begin{bmatrix}x\\\dot x\end{bmatrix}$$

因此现在我们必须设计我们的状态转移。时间步长的牛顿方程为：

$$\begin{aligned} x_t &= x_{t-1} + v\Delta t \\
 v_t &= v_{t-1}\end{aligned}$$
 
请注意，我们需要将其转换为线性方程

$$\begin{bmatrix}x\\\dot x\end{bmatrix} = \mathbf F\begin{bmatrix}x\\\dot x\end{bmatrix}$$

设置

$$\mathbf F = \begin{bmatrix}1 &\Delta t\\ 0 & 1\end{bmatrix}$$

即可得到上述方程。

最后，我们设计测量函数。测量函数需要实现

$$\mathbf z = \mathbf{Hx}$$

我们的传感器仍然只读取位置，因此它应该从状态中获取位置，并将速度和加速度清零，如下所示：

$$\mathbf H = \begin{bmatrix}1 & 0 \end{bmatrix}$$

此函数构建并返回一个一阶卡尔曼滤波器。

In [ ]:
def FirstOrderKF(R, Q, dt):
    """创建一阶KalmanFilter。将R和Q指定为浮点数。"""
    
    kf = KalmanFilter(dim_x=2, dim_z=1)
    kf.x = np.zeros(2)
    kf.P *= np.array([[100, 0], [0, 1]])
    kf.R *= R
    kf.Q = Q_discrete_white_noise(2, dt, Q)
    kf.F = np.array([[1., dt],
                     [0., 1]])
    kf.H = np.array([[1., 0]])
    return kf

### 二阶卡尔曼滤波器

二阶卡尔曼滤波器跟踪二阶系统，例如位置、速度和加速度。状态变量将是

$$ \mathbf x = \begin{bmatrix}x\\\dot x\\\ddot{x}\end{bmatrix}$$

所以现在我们必须设计我们的状态转移。牛顿方程的一个时间步长为：

$$\begin{aligned} x_t &= x_{t-1} + v_{t-1}\Delta t + 0.5a_{t-1} \Delta t^2 \\
 v_t &= v_{t-1} + a_{t-1}\Delta t \\
 a_t &= a_{t-1}\end{aligned}$$
 
请注意，我们需要将其转换为线性方程。

$$\begin{bmatrix}x\\\dot x\\\ddot{x}\end{bmatrix} = \mathbf F\begin{bmatrix}x\\\dot x\\\ddot{x}\end{bmatrix}$$

设置

$$\mathbf F = \begin{bmatrix}1 & \Delta t &.5\Delta t^2\\ 
0 & 1 & \Delta t \\
0 & 0 & 1\end{bmatrix}$$

给我们上述方程。

最后，我们设计测量函数。测量函数需要实现

$$z = \mathbf{Hx}$$

我们的传感器仍然只读取位置，因此它应该从状态中获取位置，并将速度归零，如下所示：

$$\mathbf H = \begin{bmatrix}1 & 0 & 0\end{bmatrix}$$

此函数构造并返回一个二阶卡尔曼滤波器。

In [ ]:
def SecondOrderKF(R_std, Q, dt, P=100):
    """ 创建二阶KalmanFilter。将R和Q指定为浮点数。"""
    
    kf = KalmanFilter(dim_x=3, dim_z=1)
    kf.x = np.zeros(3)
    kf.P[0, 0] = P
    kf.P[1, 1] = 1
    kf.P[2, 2] = 1
    kf.R *= R_std**2
    kf.Q = Q_discrete_white_noise(3, dt, Q)
    kf.F = np.array([[1., dt, .5*dt*dt],
                     [0., 1.,       dt],
                     [0., 0.,       1.]])
    kf.H = np.array([[1., 0., 0.]])
    return kf

## 评估滤波器阶数

现在我们可以运行每个卡尔曼滤波器并评估结果。

我们如何评估结果？我们可以通过绘制轨迹和卡尔曼滤波器输出并仔细观察结果来进行定性评估。但是，一种严格的方法使用数学。回想一下，系统协方差矩阵 $\mathbf P$ 包含每个状态变量的计算方差和协方差。对角线包含方差。请记住，如果噪声是高斯的，大约99％的所有测量值都在 $3\sigma$ 内。如果这不清楚，请在继续之前查看高斯章节，因为这是一个重要的观点。

因此，我们可以通过查看估计状态和实际状态之间的残差并将它们与从 $\mathbf P$ 推导出的标准差进行比较来评估滤波器。如果滤波器执行正确，则99％的残差都将在 $3\sigma$ 内。这对于所有状态变量都成立，而不仅仅是位置。

我必须提到这只适用于模拟系统。真实传感器不是完美的高斯分布，你可能需要扩大你的标准，例如使用真实传感器数据的$5\sigma$。

那么，让我们对我们的一阶系统运行第一阶卡尔曼滤波器，并评估其表现。你可以想象它会表现得很好，但是让我们使用标准差来看一下。

首先，让我们编写一个程序为我们生成嘈杂的测量数据。

In [ ]:
def simulate_system(Q, count):
    obj = ConstantVelocityObject(x0=.0, vel=0.5, noise_scale=Q)
    xs, zs = [], []
    for i in range(count):
        x = obj.update()
        z = sense(x)
        xs.append(x)
        zs.append(z)
    return np.array(xs), np.array(zs)

现在让我们编写一个程序来执行过滤并将输出保存在'Saver' 对象中。

In [ ]:
from filterpy.common import Saver

def filter_data(kf, zs):
    s = Saver(kf)
    kf.batch_filter(zs, saver=s)
    s.to_array()
    return s

现在我们已经准备好运行滤波器并查看结果了。

In [ ]:
from kf_book.book_plots import plot_kf_output

R, Q = 1, 0.03
xs, zs = simulate_system(Q=Q, count=50)

kf = FirstOrderKF(R, Q, dt=1)
data1 = filter_data(kf, zs)

plot_kf_output(xs, data1.x, data1.z)

看起来滤波器表现良好，但很难准确评估其性能。让我们查看残差并看看它们是否有帮助。我们将经常这样做，因此我将编写一个函数来绘制它们。

In [ ]:
from kf_book.book_plots import plot_residual_limits, set_labels

def plot_residuals(xs, data, col, title, y_label, stds=1):
    res = xs - data.x[:, col]
    plt.plot(res)
    plot_residual_limits(data.P[:, col, col], stds)
    set_labels(title, 'time (sec)', y_label)

In [ ]:
plot_residuals(xs[:, 0], data1, 0, 
               title='一阶位置residual(1$\sigma$)',
               y_label='米')   

我们如何解释这张图呢？残差被画成折线 - 表示测量值和卡尔曼滤波器预测位置之间的差异。如果没有测量噪声且卡尔曼滤波器的预测始终完美，残差将始终为零。因此，理想输出应该是0的水平线。我们可以看到残差围绕着0中心，这使我们相信噪声是高斯的（因为误差在0上下均匀分布）。虚线之间的黄色区域展示了1个标准偏差的滤波器理论性能。换句话说，大约68%的误差应该落在虚线之内。残差在这个范围之内，因此我们可以看出滤波器表现良好，没有发散。

让我们看一下速度的残差。

In [ ]:
plot_residuals(xs[:, 1], data1, 1, 
               title='一阶速度residual(1$\sigma$)',
               y_label='米/秒')   

再次验证，残差落在滤波器的理论性能范围内，因此我们有信心该滤波器设计良好适用于该系统。

现在我们使用零阶卡尔曼滤波器再次进行相同的操作。所有的代码和数学方法基本相同，因此我们只看结果，不讨论实现细节。

In [ ]:
kf0 = ZeroOrderKF(R, Q)
data0 = filter_data(kf0, zs)
plot_kf_output(xs, data0.x, data0.z)

正如我们所预期的那样，滤波器存在问题。回想一下 g-h 滤波器，我们将加速度纳入系统中。由于没有足够的项允许滤波器快速调整到速度的变化，所以 g-h 滤波器总是滞后于输入。在每个 `predict()` 步骤中，卡尔曼滤波器都假设位置没有变化 - 如果当前位置是 4.3，则会预测下一个时间段的位置也是 4.3。当然，实际位置更接近于 5.3。带噪声的测量值可能是 5.4，因此滤波器选择介于 4.3 和 5.4 之间的估计值，导致它严重滞后于实际值 5.3。这在下一步、下一步等等中也会发生。滤波器永远无法追上实际值。

这引出了一个非常重要的观点。"常数"的假设仅适用于离散样本之间的恒定性假设。滤波器的输出仍然可以随时间变化。

现在让我们看看残差。我们没有跟踪速度，因此只能查看位置的残差。

In [ ]:
plot_residuals(xs[:, 0], data0, 0, 
               title='零阶位置residual（3$\sigma$）',
               y_label='米',
               stds=3)

我们可以看到，滤波器几乎立即发散。几秒钟后，残差超过了三个标准差的界限。重要的是要理解，协方差矩阵 $\mathbf P$ 只报告了滤波器的 *理论* 性能，*假设* 所有输入都是正确的。换句话说，这个卡尔曼滤波器正在发散，但 $\mathbf P$ 暗示着卡尔曼滤波器的估计结果随着时间的推移越来越好，因为方差越来越小。滤波器无法知道您对系统的状态进行了欺骗。这有时被称为 *自以为是的* 滤波器-它对其性能过于自信。

在这个系统中，发散是立即且显著的。在许多系统中，它只会逐渐而且轻微地出现。查看这样的图表以确保滤波器的性能在其理论性能的范围内非常重要。

现在让我们尝试一个二阶系统。这可能是一个好主意。毕竟，我们知道模拟物体的运动中有一些噪音，这意味着存在一些加速度。为什么不用二阶模型来模拟加速度呢？如果没有加速度，加速度应该被估计为0，对吗？但是这是否是发生的呢？在继续之前先思考一下。

In [ ]:
kf2 = SecondOrderKF(R, Q, dt=1)
data2 = filter_data(kf2, zs)
plot_kf_output(xs, data2.x, data2.z)

这个表现和你预期的一样吗？

我们可以看到，与一阶滤波器相比，二阶滤波器的性能较差。为什么？这个滤波器模拟加速度，因此测量值中的大变化被解释为加速度而不是噪声。因此，滤波器紧密跟踪噪声。不仅如此，如果噪声始终高于或低于轨道，滤波器还会在某些位置*超调*噪声，因为滤波器错误地假设存在一个不存在的加速度，因此在每个测量中，其预测越来越远离轨道。这不是一个好的情况。

不过，轨道看起来并不*太差*。现在，我们来看看残差所告诉我们的故事。我在这里加了一个插曲。对于二阶系统，残差看起来并不可怕，因为它们没有发散或超过三个标准差。然而，查看一阶滤波器和二阶滤波器的残差非常有意义，因此我在同一张图上绘制了两者。

In [ ]:
res2 = xs[:, 0] - data2.x[:, 0]
res1 = xs[:, 0] - data1.x[:, 0]

plt.plot(res2, label='二阶')
plt.plot(res1, ls="--", label='一阶')
plot_residual_limits(data2.P[:, 0, 0])
set_labels('二阶位置residual',
           '米', '时间（秒）')
plt.legend();



二阶位置残差略差于一阶滤波器的残差，但仍落在滤波器的理论极限内。这里没有什么特别警报的。

现在让我们看看速度的残差。

In [ ]:
res2 = xs[:, 1] - data2.x[:, 1]
res1 = xs[:, 1] - data1.x[:, 1]

plt.plot(res2, label='二阶')
plt.plot(res1, ls='--', label='一阶')
plot_residual_limits(data2.P[:, 1, 1])
set_labels('二阶速度residual', 
                      '米/秒', '时间（秒）')
plt.legend();

这里的情况非常不同。尽管二阶系统的残差落在滤波器性能的理论范围内，但我们可以看到残差比一阶滤波器的残差*要差得多*。这是这种情况的常见结果。滤波器假设存在加速度，而实际上并没有。它将测量中的噪声误认为加速度，并在每个预测周期中将其添加到速度估计中。当然，实际上并没有加速度，因此速度的残差比其最优值大得多。

我还有一个招数。我们有一个一阶系统;即速度多少是恒定的。现实世界中的系统从来都不是完美的，因此速度在时间段之间从来不完全相同。当我们使用一阶滤波器时，我们考虑到了速度的微小变化，这是由*过程噪声*引起的。矩阵$\mathbf Q$被计算用于考虑这种微小变化。如果我们转向二阶滤波器，我们现在正在考虑速度的变化。也许现在我们没有过程噪声，我们可以将$\mathbf Q$设置为零！

In [ ]:
kf2 = SecondOrderKF(R, 0, dt=1)
data2 = filter_data(kf2, zs)
plot_kf_output(xs, data2.x, data2.z)

在我看来，滤波器似乎很快就会收敛到实际轨迹。成功！

或者，也许不是。将过程噪声设置为0告诉滤波器过程模型是完美的。让我们看看滤波器在更长时间内的表现。

In [ ]:
np.random.seed(25944)
xs500, zs500 = simulate_system(Q=Q, count=500)

kf2 = SecondOrderKF(R, 0, dt=1)
data500 = filter_data(kf2, zs500)

plot_kf_output(xs500, data500.x, data500.z)
plot_residuals(xs500[:, 0], data500, 0, 
               '二阶位置residual',
               '米') 

我们可以看到滤波器的性能非常糟糕。在轨迹图中，我们可以看到滤波器在一段时间内偏离轨迹。残差图使问题更加明显。在第100次更新之前，滤波器与理论性能急剧分歧。它 *可能* 在最后收敛，但我不确定。整个时间里，滤波器报告越来越小的方差。**不要相信滤波器的协方差矩阵告诉你滤波器的性能如何**！

为什么会这样呢？回想一下，如果我们将过程噪声设为零，我们就是在告诉滤波器只使用过程模型。测量结果最终会被忽略。物理系统并不完美，因此滤波器无法适应这种不完美的行为。

也许只需要一个非常低的过程噪声？我们来试试。

In [ ]:
np.random.seed(32594)
xs2000, zs2000 = simulate_system(Q=0.0001, count=2000)

kf2 = SecondOrderKF(R, 0, dt=1)
data2000 = filter_data(kf2, zs2000)

plot_kf_output(xs2000, data2000.x, data2000.z)
plot_residuals(xs2000[:, 0], data2000, 0, 
               'Second Order Position Residuals',
               'meters') 

同样，残差图告诉了我们这个故事。轨迹看起来非常好，但残差图显示滤波器在相当长的时间内出现了发散的情况。

如何理解这些内容呢？你可能会认为最后一个图对你的应用来说已经足够好了，也许确实如此。然而，我要警告你的是，一个发散的滤波器并不总是会收敛。在不同的数据集或表现不同的物理系统中，你可能会得到一个越来越远离测量值的滤波器。

此外，让我们从数据拟合的角度来考虑这个问题。假设我给你两个点，并告诉你要在这些点上拟合一条直线。

In [ ]:
plt.scatter([1, 2], [1, 1], s=100, c='r')
plt.plot([0, 3], [1, 1]);

一条直线是唯一可能的答案。此外，这个答案是最优的。如果我给你更多的点，你可以使用最小二乘拟合来找到最佳的直线，而答案仍然是在最小二乘意义下最优的。

但是假设我告诉你要将这两个点拟合成更高阶的多项式。问题就有无限多个答案。例如，无限多个二次抛物线都可以通过这些点。当Kalman滤波器的阶数高于物理过程时，它也有无限多个解可供选择。答案不仅不是最优的，而且通常会发散并且永远无法恢复。

为了获得最佳性能，您需要一个订单与系统订单相匹配的滤波器。在许多情况下，这很容易做到 - 如果您正在设计一个卡尔曼滤波器来读取冰箱的温度计，则似乎清楚地选择了零阶滤波器。但是，如果我们正在跟踪一辆汽车，应该使用什么订单？当汽车以恒定速度直线行驶时，一阶序列将表现良好，但是汽车会转弯，加速和减速，在这种情况下，二阶滤波器将表现更好。这是在自适应滤波器一章中解决的问题。在那里，我们将学习如何设计一个滤波器，以适应跟踪对象行为中的订单变化。

话虽如此，只要添加足够的过程噪声并且保持离散化周期较小（通常在每秒100个样本处局部线性），低阶滤波器就可以跟踪更高阶的过程。结果可能不是最佳的，但仍然可以非常好，我总是首选这个工具，而不是尝试自适应滤波器。让我们以加速度为例进行模拟。首先，进行模拟。

In [ ]:
class ConstantAccelerationObject(object):
    def __init__(self, x0=0, vel=1., acc=0.1, acc_noise=.1):
        self.x = x0
        self.vel = vel
        self.acc = acc
        self.acc_noise_scale = acc_noise
    
    def update(self):
        self.acc += randn() * self.acc_noise_scale       
        self.vel += self.acc
        self.x += self.vel
        return (self.x, self.vel, self.acc)
  
R, Q = 6., 0.02
def simulate_acc_system(R, Q, count):
    obj = ConstantAccelerationObject(acc_noise=Q)
    zs = []
    xs = []
    for i in range(count):
        x = obj.update()
        z = sense(x, R)
        xs.append(x)
        zs.append(z)
    return np.asarray(xs), zs

np.random.seed(124)
xs, zs = simulate_acc_system(R=R, Q=Q, count=80)
plt.plot(xs[:, 0]);


现在我们将使用一个二阶滤波器对数据进行滤波。

In [ ]:
np.random.seed(124)
xs, zs = simulate_acc_system(R=R, Q=Q, count=80)

kf2 = SecondOrderKF(R, Q, dt=1)
data2 = filter_data(kf2, zs)

plot_kf_output(xs, data2.x, data2.z, aspect_equal=False)
plot_residuals(xs[:, 0], data2, 0, 
               '二阶位置残差',
               '米')

In [ ]:

我们可以看到滤波器在理论限制内运行。

现在让我们使用一个低阶滤波器。如已经演示的那样，低阶滤波器会滞后信号，因为它没有建模加速度。然而，我们可以通过增加过程noise的大小来在一定程度上解决这个问题。滤波器会将加速度视为过程模型中的noise。结果会是次优的，但如果设计得好，它不会发散。选择额外的过程noise量并不是一门确切的科学。您将不得不使用代表性data进行实验。在这里，我将其乘以10，并获得了良好的结果。

```python
kf3 = FirstOrderKF(R, Q * 10, dt=1)
data3= filter_data(kf3, zs)

plot_kf_output(xs，data3.x，data3.z，aspect_equal=False)
plot_residuals(xs [:，0]，data3，0，'一阶位置残差'，'米')

In [ ]:

想想如果你将过程noise放大多次，会发生什么。大的过程noise告诉滤波器偏向于测量，因此我们预计滤波器会紧密地模仿测量中的noise。让我们来看看。

```python
kf4 = FirstOrderKF(R，Q * 10000，dt=1)
data4 = filter_data(kf4，zs)

plot_kf_output(xs，data4.x，data4.z，aspect_equal=False)
plot_residuals(xs [:，0]，data4，0，'一阶位置residual'，'米') 

## 练习：状态变量设计

正如我已经提到的，您可以按任意顺序放置 $\mathbf x$ 中的变量。例如，您可以将一维恒定加速度定义为 $\mathbf x = \begin{bmatrix}\ddot x & x & \dot x\end{bmatrix}^\mathsf T$。我无法想象为什么您要选择那个顺序，但这是可能的。

让我们做一些更合理的事情。为在二维空间中移动的机器人设计一个二阶滤波器，其中 $\mathbf x = \begin{bmatrix}x & y & \dot x & \dot y \end{bmatrix}^\mathsf T$。在本章中，我们一直在使用 $\mathbf x = \begin{bmatrix}x & \dot x & y & \dot y \end{bmatrix}^\mathsf T$。

为什么要选择不同的顺序？正如您将在接下来看到的，更改 $\mathbf x$ 的顺序会更改大部分滤波器矩阵的顺序。根据您想要检查的数据（例如 $\mathbf P$ 中的相关性），$\mathbf x$ 的各种排列可以使这变得更容易或更困难。

考虑如何做到这一点。需要改变什么？显然，您只需要更改卡尔曼滤波器矩阵以反映这个新设计。

使用以下模板代码尝试一下：

In [ ]:

N = 30 # 迭代次数
dt = 1.0 # 时间步长
R_std = 0.35
Q_std = 0.04

sensor = PosSensor((0, 0), (2, .5), noise标准差=R_std)
zs = np.array([sensor.read() for _ in range(N)])

tracker = KalmanFilter(dim_x=4, dim_z=2)
# 在此处分配状态变量

xs, ys = [], []
for z in zs:
    tracker.predict()
    tracker.update(z)
    xs.append(tracker.x[0])
    ys.append(tracker.x[1])
plt.plot(xs, ys);

### 解决方案

让我们从 $\mathbf F$ 开始。经过一些练习，您应该能够直接编写矩阵。如果您发现难以做到这一点，则按照状态变量使用的变量顺序编写 $\mathbf F$ 的方程组即可。

$$
x = 1x + 0y + 1\dot x\Delta t + 0 \dot y\Delta t \\
y = 0x + 1y + 0\dot x\Delta t + 1 \dot y\Delta t \\
\dot x = 0x + 0y + 1\dot x\Delta t + 0 \dot y\Delta t \\
\dot y = 0x + 0y + 0\dot x\Delta t + 1 \dot y\Delta t 
$$

我们可以将系数复制出来得到

$$\mathbf F = \begin{bmatrix}1&0&\Delta t & 0\\0&1&0&\Delta t\\0&0&1&0\\0&0&0&1\end{bmatrix}$$

状态噪声也需要重新排列。首先，找出它应该如何排列以适应状态变量的顺序。如果你将状态变量垂直和水平地写在矩阵上，就可以更容易地看出它们如何配对。在Jupyter笔记本中这样做很困难，所以我将在这里放弃。

$$\mathbf Q = 
\begin{bmatrix}
\sigma_x^2 & \sigma_{xy} & \sigma_{x\dot x} & \sigma_{x\dot y} \\
\sigma_{yx} & \sigma_y^2 & \sigma_{y\dot x} & \sigma_{y\dot y} \\
\sigma_{\dot x x} & \sigma_{\dot x y} & \sigma_{\dot x}^2 & \sigma_{\dot x \dot y} \\
\sigma_{\dot y x} & \sigma_{\dot y y} & \sigma_{\dot y \dot x} & \sigma_{\dot y}^2
\end{bmatrix}
$$

由于$x$和$y$之间不存在相关性，因此我们可以将它们的相关项设为零

$$\mathbf Q = 
\begin{bmatrix}
\sigma_x^2 & 0 & \sigma_{x\dot x} & 0 \\
0 & \sigma_y^2 & 0 & \sigma_{y\dot y} \\
\sigma_{\dot x x} & 0 & \sigma_{\dot x}^2 & 0 \\
0 & \sigma_{\dot y y} & 0 & \sigma_{\dot y}^2
\end{bmatrix}
$$

现在，您可以看到这个模式并且也许能更快地设计$\mathbf Q$。

`Q_discrete_white_noise`会生成一个有不同顺序的矩阵，但我们可以从中复制术语，我们将在下面的代码中看到。

现在让我们设计 $\mathbf H$。它将状态 $\begin{bmatrix}x & y & \dot x & \dot y \end{bmatrix}^\mathsf T$ 转换为测量值 $\mathbf z = \begin{bmatrix}z_x & z_y\end{bmatrix}^\mathsf T$。

$$
\begin{aligned}
\mathbf{Hx} &= \mathbf z \\
\begin{bmatrix}?&?&?&?\\?&?&?&?\end{bmatrix}\begin{bmatrix}x \\ y \\ \dot x \\ \dot y \end{bmatrix} &= \begin{bmatrix}z_x \\ z_y\end{bmatrix}
\end{aligned}
$$

现在填充矩阵应该很容易：
$$
\begin{bmatrix}1&0&0&0\\0&1&0&0\end{bmatrix}\begin{bmatrix}x \\ y \\ \dot x \\ \dot y \end{bmatrix} = \begin{bmatrix}z_x \\ z_y\end{bmatrix}
$$

测量值 $\mathbf z = \begin{bmatrix}z_x & z_y\end{bmatrix}^\mathsf T$ 没有改变，所以 $\mathbf R$ 不变。

最后，$\mathbf P$。它使用与 $\mathbf Q$ 相同的排序，因此已经为我们设计好了。

In [ ]:
N = 30 # 迭代次数
dt = 1.0 # 时间步长
R_std = 0.35
Q_std = 0.04

M_TO_FT = 1 / 0.3048

sensor = PosSensor((0, 0), (2, .5), noise_std=R_std)
zs = np.array([sensor.read() for _ in range(N)])

tracker = KalmanFilter(dim_x=4, dim_z=2)

tracker.F = np.array([[1, 0, dt,  0],
                      [0, 1,  0, dt],
                      [0, 0,  1,  0],
                      [0, 0,  0,  1]])

tracker.H = np.array([[M_TO_FT, 0, 0, 0],
                      [0, M_TO_FT, 0, 0]])

tracker.R = np.eye(2) * R_std**2
q = Q_discrete_white_noise(dim=2, dt=dt, var=Q_std**2)
tracker.Q[0,0] = q[0,0]
tracker.Q[1,1] = q[0,0]
tracker.Q[2,2] = q[1,1]
tracker.Q[3,3] = q[1,1]
tracker.Q[0,2] = q[0,1]
tracker.Q[2,0] = q[0,1]
tracker.Q[1,3] = q[0,1]
tracker.Q[3,1] = q[0,1]

tracker.x = np.array([[0, 0, 0, 0]]).T
tracker.P = np.eye(4) * 500.

xs, ys = [], []
for z in zs:
    tracker.predict()
    tracker.update(z)
    xs.append(tracker.x[0])
    ys.append(tracker.x[1])
plt.plot(xs, ys);

## 检测和拒绝错误测量

KalmanFilter无法检测和拒绝错误的测量。假设你正在跟踪飞机，并接收到距离飞机当前位置100公里的测量data。如果你使用该值调用更新函数，则新的估计值将被极端偏向于该测量值。

我将运行一个模拟来给出一个具体的例子。经过100个时间步，我将使用一个等于当前位置两倍的测量值进行更新。`filterpy.common`提供`kinematic_kf`，它可以创建任意维度和阶数的线性运动学滤波器。我在这里使用它来使代码整洁，但在本书的其余部分中我没有使用它，因为我希望你多练习编写滤波器。

```python
from filterpy.common import kinematic_kf

kf = kinematic_kf(dim=2, order=1, dt=1.0, order_by_dim=False)
kf.Q = np.diag([0, 0, .003, .003])
kf.x = np.array([[1., 1., 0., 0.]]).T
kf.R = np.diag([0.03, 0.21]) # 使用不同的误差

In [ ]:
for i in range(101):
    kf.predict()
    kf.update(np.array([[i*.05, i*.05]])) # around 200 kph

p0 = kf.x[0:2]

kf.predict()
prior = kf.x
z = kf.x[0:2]*2
kf.update(z)
p1 = kf.x[0:2]

# 计算先验测量误差
y = np.abs(z - kf.H @ prior)
dist = np.linalg.norm(y)

np.set_printoptions(precision=2, suppress=True)

print(f'坏的测量值       : {z.T} 公里')
print(f'坏测量之前的估计值: {p0.T} 公里')
print(f'坏测量之后的估计值 : {p1.T} 公里')
print(f'估计值平移        : {np.linalg.norm(p1 - prior[:2]):.1f} 公里')
print(f'距离先验估计值的距离   : {dist:.1f} 公里')

`kinematic_kf`？那是什么？ `filterpy.common` 提供了 `kinematic_kf`，使您可以创建任意维度和阶数的线性运动学滤波器。 我在本书中没有使用它，因为我希望您能够获得大量创建 Kalman 滤波器的经验。 我在这里使用它只是为了使我的示例简短并让您接触到库的这个部分。

回到主题。您可以看到，估计值跳了3.4公里，而预测值（prior）和测量值之间的误差超过了7公里。

我们可以怎么做来避免这种情况？我们的第一个想法可能是添加一个检查，如果prior与测量值相距很远就停止。为什么是prior而不是当前的估计值？因为在更新之后，估计值现在可能相当接近错误的测量值，尽管在这种情况下并非如此。

请注意，虽然我可以只写 `prior [0：2] - z`来获取误差，但我使用了数学上正确的 $\mathbf z - \mathbf {Hx}$ 来计算误差。这只是为了说明；Kalman滤波器类将创新存储在 `KalmanFilter.y` 中。我使用它来说明上面计算的值，如下所示：

In [ ]:
print(f'error = {np.linalg.norm(kf.y):.1f} km, at a speed of {dist*3600:.0f} kph')

在这个例子中，测量值距离预测位置近7公里。这听起来很“远”。是吗？如果单位是公里，更新速率是1秒，那么这个误差意味着没有飞机可以飞行超过25000千米每小时。如果单位是厘米，时间间隔是1分钟，那么这个误差可能就非常小了。

我们可以添加一个检查，考虑飞机的性能限制：

In [ ]:
vel = y / dt
if vel >= MIN_AC_VELOCITY and vel <= MAX_AC_VELOCITY:
    kf.update()

你认为这是一个合理和健壮的解决方案吗？在阅读后，请提出尽可能多的反对意见。

这对我来说并不令人满意。假设我们只是用猜测的位置初始化了滤波器；我们会丢弃好的测量结果，从而无法开始滤波。其次，这忽略了我们对传感器和过程误差的了解。卡尔曼滤波器在 $\mathbf P$ 中维护其当前精度。如果 $\mathbf P$ 意味着 $\sigma_x = 10$ 米，而测量值为 1 千米，很明显该测量值是错误的，因为它比先前的值偏离了 100 个标准差。

让我们绘制 $\mathbf P$。我将绘制第一、二和三个标准差。

In [ ]:
x, P = kf.x[0:2], kf.P[0:2, 0:2]
plot_covariance_ellipse(x, P, std=[1,2,3])

在之前的章节中，$\mathbf P$ 是一个圆，而不是一个椭圆。在我的代码中，我设置了 $\mathbf R = \bigl[ \begin{smallmatrix}0.03 & 0 \\ 0 & 0.15\end{smallmatrix}\bigl ]$，这使得 $y$ 的测量误差是 $x$ 的 5 倍。这有些人为，但在随后的章节中，几乎所有的协方差都是椭圆形的，这是有充分理由的。

想想这意味着什么。统计学告诉我们，99% 的所有测量值都在 3 个标准差内；这意味着 99% 的所有测量值应该在这个椭圆内。让我们用椭圆来绘制这个测量值。

In [ ]:
plot_covariance_ellipse(x, P, std=[1,2,3])
plt.scatter(z[0], z[1], marker='x');

很明显，这个测量值远远超出了先验的协方差范围；我们可能希望将其视为一个错误的测量值，不使用它。我们该如何做到这一点？

首先的想法是提取 $x$ 和 $y$ 的标准差，然后编写一个简单的 if 语句。在这里，我将使用 `KalmanFilter` 类的另一个特性。`residual_of` 方法计算与先前的残差。在这种情况下，我不需要使用它，因为 `kf.y` 已经通过调用 `update()` 分配了，但是如果我们丢弃测量值，则尚未调用 `update()`，并且 `kf.y` 将包含来自上一个时期的创新。

首先，让我们介绍两个术语。我们正在讨论 **gating**。**gate** 是一个公式或算法，用于确定测量的好坏。只有好的测量才能通过 gate。这个过程被称为 gating。

在实践中，测量值不是纯高斯分布的，因此 3 个标准差的 gate 很可能会丢弃一些好的测量值。我很快会详细说明，现在我们将使用 4 个标准差。

In [ ]:
GATE_LIMIT = 4.
std_x = np.sqrt(P[0,0])
std_y = np.sqrt(P[1,1])
y = kf.residual_of(z)[:,0]

if y[0] > GATE_LIMIT * std_x or y[1] > GATE_LIMIT * std_y:
    print(f'放弃该测量, 误差为{y[0]/std_x:.0f}标准差, {y[1]/std_y:.0f}标准差')
    
print('y为', y)
print(f'标准差为{std_x:.2f} {std_y:.2f}')

我们发现误差大约相差39和18个标准偏差。这样就足够好了吗？

也许。但是请注意，if语句形成了一个包围椭圆的矩形区域。在下面的图中，我画了一个明显在3个标准差椭圆之外的测量结果，但会被接受，并且另一个位于3个标准差边界上的测量结果。

In [ ]:
plot_covariance_ellipse(x, P, std=[1,2,3])
plt.scatter(8.08, 7.7, marker='x')
plt.scatter(8.2, 7.65, marker='x');

有替代定义此门的方法。**马氏距离（mahalanobis distance）**是从分布到一个点的统计距离度量。在我们深入定义和数学内容之前，让我们计算一些点的马氏距离。`filterpy.stats` 实现了 `mahalanobis()`。

In [ ]:
from filterpy.stats import mahalanobis
m = mahalanobis(x=z, mean=x, cov=P)
print(f'马氏距离 = {m:.1f}')

我们不知道单位，但我们可以将其与在 $x$ 和 $y$ 中分别计算出的标准差误差 39 和 18 进行比较，可以看出它是相当接近的。让我们看看上面绘制的点的马氏距离。

In [ ]:
print(f'马氏距离 = {mahalanobis(x=[8.08, 7.7], mean=x, cov=P):.1f}')
print(f'马氏距离 = {mahalanobis(x=[8.2, 7.65], mean=x, cov=P):.1f}')

正如我们所看到的，马氏距离计算标量标准差“距离”点到分布，就像欧几里得距离计算点到另一个点的标量距离一样。

上面的单元格证明了这一点。坐落在3个标准差边界上的点具有3.0的马氏距离，而椭圆外的点具有3.6个标准差的值。

我们如何计算马氏距离？定义如下：

$$D_m= \sqrt{(\mathbf x-\mu)^\mathsf T \mathbf S^{-1} (\mathbf x-\mu)}$$

请注意，这与欧几里得距离非常相似，我们可以将其写成：

$$D_e= \sqrt{(\mathbf x-\mathbf y)^\mathsf T (\mathbf x-\mathbf y)}$$

实际上，如果协方差 $\mathbf S$ 是单位矩阵，马氏距离就与欧氏距离相同。你应该能够看出它在线性代数中是如何工作的：单位矩阵的逆矩阵是单位矩阵，因此我们实际上是在用 1 乘以这些项。从直觉上考虑。如果每个维度中的标准差为 1，则在均值周围半径为 1 的圆上的任何点都将位于 1 标准差圆上，并且在欧几里得度量中也相距 1 个单位。

这提出了另一种解释。如果协方差矩阵是对角线矩阵，那么我们可以将马氏距离视为 *缩放的* 欧几里得距离，其中对角线上的每个术语都按对角线上的协方差进行缩放。

$$D_m = \sqrt{\sum_{i-1}^N \frac{(x_i - \mu_i)^2}{\sigma_i}}$$

在二维中，它将是

$$D_m = \sqrt {\frac{1}{\sigma_x^2}(x_0 - x_1)^2 + \frac{1}{\sigma_y^2}(y_0 - y_1)^2}$$

这应该让你了解马氏距离方程。你不能除以一个矩阵，但在手动计算时，乘以逆矩阵基本上是相同的，这是一种概括性的说法。在每一侧上乘以差异 $\mathbf y = \mathbf x - \mathbf \mu$ 给我们缩放协方差的平方范数：$\mathbf y^\mathsf T \mathbf S^{-1}\mathbf y^\mathsf T$。协方差项都是平方的，因此在最后取平方根得到一个标量距离，即欧几里得距离乘以协方差。

### 门控和数据关联策略

上述两个门在一些文献中被称为矩形门和椭圆门，因为它们的形状不同。这里还有更多的选择，我不会在这里探讨。例如，机动门用于定义一个区域，该区域内物体可以进行机动，考虑到物体当前的速度和机动能力。例如，战斗机的机动门形状类似于锥体，投射在其当前行进方向的前面。汽车的门形状则更小，是一个二维扇形，投射在汽车前方。船的门形状则更窄，因为它的转向或快速加速能力很小。

你应该使用哪种选关技术？没有一个单一的答案。很大程度上取决于问题的维度和你拥有的计算能力。矩形门非常便宜，Maneuver门不差，但在更高的维度下，椭圆门可能会很昂贵。然而，随着维度的增加，椭圆体和矩形门之间的相对面积差异显著增加。

这比你想象的更重要。每个测量都有噪声。虚假测量可能会落在我们的门内，导致我们接受它。超出椭圆体范围的面积越大，门通过错误测量的概率就越大。我不会在这里进行数学计算，但在5维中，矩形门比椭圆体门多两倍的可能性接受错误测量。

如果计算时间是个问题，并且你有许多虚假的测量数据，你可以采用两个门的方法。第一个门是大而矩形的，用作第一次筛选，以去除明显不好的测量数据。通过第一个门的少数测量数据，随后需要进行更昂贵的马氏距离计算。如果你在现代台式机处理器上运行，那么这些矩阵乘法的时间并不重要，但如果你在性能较差的嵌入式芯片上运行，则可能会有影响。

数据关联是一个需要自己的书籍来讨论的广泛话题。典型的例子是雷达跟踪。雷达在每次扫描中检测到多个信号。从这些信号中，我们需要形成飞行器轨迹，并排除噪声测量。这是一个非常困难的问题。假设第一次扫描得到5个测量值。我们将创建5个潜在的轨迹。在下一次扫描中，我们得到了6个测量值。我们可以将第一个测量值与第二个测量值中的任何一个相结合，这给了我们30个潜在的轨迹。但是，也有可能这些都是我们上次扫描没有看到的新的飞行器，这将给我们带来6个新的轨迹。在几个时期之后，我们将达到数百万，甚至数十亿个潜在的轨迹。

有了这个数十亿条轨迹的列表，我们就可以计算每个轨迹的得分。我将在下一节中提供数学计算方法。但是，想象一下一个在3个时段内形成“Z”形的轨迹。没有飞行器可以像那样机动，所以我们会给它非常低的真实可能性。另一个轨迹形成一条直线，但推定速度为10,000公里每小时。这也非常不可能。另一个轨迹以200公里每小时的速度弯曲。这有很高的可能性。

因此，跟踪变成了门控、数据关联和修剪的问题。例如，假设第二次雷达扫描刚刚发生。我应该将所有可能的组合合并成轨迹吗？我可能不应该这样做。如果点1、扫描1插入了一个200公里每小时的速度，点3、扫描2我们应该从中形成一个轨迹。如果速度是5000公里每小时，我们不应该费心；我们知道这个轨迹是如此不可能，以至于不存在。然后，随着轨迹的增长，我们将为它们定义良好的椭圆形或机动门，我们可以更加选择地将测量与轨迹相关联。

有关联的方案。我们可以选择将测量仅与一个轨迹相关联。或者，我们可以选择将测量与多个轨迹相关联，反映了我们对其所属轨迹的不确定性。例如，从雷达的角度来看，飞机轨迹可能会交叉。随着飞机的接近，将单个测量与其中一个飞机相关联可能变得不确定。您可以短暂地将测量分配给两个轨迹。随着您收集更多测量数据，您可以根据新信息的概率更高的轨迹重新分配测量数据。

“数十亿”远远无法涵盖这意味着的组合爆炸。仅仅几秒钟后，计算机内存就会不足，再长一点，您就需要使用宇宙中的每个原子来表示所有潜在的轨迹。因此，实用的算法需要积极修剪轨迹。这种修剪需要额外的计算能力。

书中的后续章节提供了现代解决这个问题的答案——*粒子滤波器*，它通过统计抽样来解决组合爆炸问题。这是我对这个问题的首选解决方案，因此我不会在本节讨论中再写更多有关技术的内容。我并不完全了解这个领域的最新研究，因此如果您需要解决需要跟踪多个对象或处理多个虚假测量的问题，请自行进行研究。粒子滤波器也有它们自己的困难和限制。

我会向您介绍一些书籍和研究者。Samuel S. Blackman的"Multiple-Target Tracking with Radar Application"是我读过的关于这个主题最清晰的书，尽管它有些过时（1986年）。Yaakov Bar-Shalom在这个领域已经写了很多。Subhash Challa等人的"Fundamentals of Object Tracking"是一部相当现代的作品，涵盖了各种方法。这本书在数学上相当严格，滤波器被呈现为代表各种贝叶斯公式的积分集合，您需要将其转化为工作算法。如果您已经掌握了本书中的所有数学知识，那么这本书是可读的，但并不容易。Lawrence D. Stone的"Bayesian Multiple Target Tracking"将其视为贝叶斯推断问题，就像我一样，但也相当理论化，您会被轻松地告知要找到一个复杂积分的最大值，而在实践中，您可能会使用粒子滤波器来解决这个问题。

回到我们的简单问题 - 跟踪一个偶尔出现错误测量的单个对象。应该如何实现？这很简单；如果测量值不好，就将其丢弃，不要调用更新。这将导致您连续调用`predict()`两次，这是可以接受的。您的不确定性会增长，但是几次错过的更新通常不会造成问题。

您应该使用什么截止值来设置门限？我不知道。理论上应该是3个标准差，但实践表明不是这样的。您需要进行实验。收集数据，使用各种门限对其进行滤波，看看哪个值能够给出最佳结果。在接下来的部分中，我会给您一些数学来评估滤波器的性能。也许您会发现需要接受所有小于4.5个标准差的测量值。我看过NASA的一个视频，他们声称使用了大约5-6个标准差的门限。这取决于您的问题和数据。

## 评估滤波器性能

设计一个卡尔曼滤波器来处理模拟的情况是很容易的。你知道在你的过程模型中注入了多少噪声，所以你可以指定 $\mathbf Q$ 具有相同的值。你也知道测量模拟中有多少噪声，所以测量噪声矩阵 $\mathbf R$ 同样很容易定义。

在实践中，设计更多的是临时性的。实际传感器很少按规格执行，它们也很少以高斯方式执行。它们也很容易被环境噪声所误导。例如，电路噪声会导致电压波动，从而影响传感器的输出。创建过程模型和噪声更加困难。对汽车进行建模非常困难。转向会导致非线性行为，轮胎会打滑，人们会强烈刹车和加速，从而导致轮胎打滑，风会使汽车偏离航线。最终结果是卡尔曼滤波器是系统的一个*不精确*模型。这种不精确性会导致次优行为，最坏的情况下会导致滤波器完全发散。

由于未知因素，您将无法分析地计算出滤波器矩阵的正确值。您将首先进行最佳估计，然后针对各种模拟和实际数据测试您的滤波器。您对性能的评估将指导您需要对矩阵进行哪些更改。我们已经完成了其中的一些工作——我已向您展示了 $\mathbf Q$ 太大或太小的效果。

现在让我们考虑更多分析性能的方法。如果卡尔曼滤波器的性能最优，其估计误差（真实状态和估计状态之间的差异）将具有以下特性：

In [ ]:
1. 估计误差的平均值为零
2. 其协方差由KalmanFilter的协方差矩阵描述

### 归一化估计误差平方（NEES）

第一种方法是最强大的，但仅在模拟中可能。如果您正在模拟一个系统，您知道其真实值或“基本事实”。然后很容易计算系统在任何一步的误差，即基本事实($\mathbf x$)和滤波器状态估计($\hat{\mathbf x}$)之间的差异：

$$\tilde{\mathbf x} = \mathbf x - \hat{\mathbf x}$$

然后我们可以定义*归一化估计误差平方* (NEES)：

$$\epsilon = \tilde{\mathbf x}^\mathsf T\mathbf P^{-1}\tilde{\mathbf x}$$

为了理解这个方程，让我们看看如果状态的维数为1时它是什么。在这种情况下，x和P都是标量，因此

$$\epsilon = \frac{x^2}{P}$$

如果这不是很清楚，请回想一下如果 $a$ 是标量，则 $a^\mathsf T=a$，并且 $a^{-1} =\frac{1}{a}$。

当协方差矩阵变小时，对于相同的误差，NEES会变大。协方差矩阵是滤波器对其误差的估计，因此，如果相对于估计误差很小，那么它的性能就比相对于相同的估计误差很大时差。

这个计算给我们一个标量结果。如果 $\mathbf x$ 是维度为 ($n \times 1$)，那么计算的维度为 ($1 \times n$)($n \times n$)($n \times 1$) = ($1 \times 1$)。我们用这个数字做什么？

这个数学问题超出了本书的范围，但是以 $\tilde{\mathbf x}^\mathsf T\mathbf P^{-1}\tilde{\mathbf x}$ 形式的随机变量被称为*自由度为n的卡方分布*，因此，序列的期望值应为 $n$。Bar-Shalom [1] 对这个主题有很好的讨论。

换言之，取所有 NEES 值的平均值，它们应该小于 x 的维度。让我们使用本章早期的一个例子来说明这一点：

In [ ]:
from scipy.linalg import inv

def NEES(xs, est_xs, Ps):
    est_err = xs - est_xs
    err = []
    for x, p in zip(est_err, Ps):
        err.append(x.T @ inv(p) @ x)
    return err

In [ ]:
R, Q = 6., 0.02
xs, zs = simulate_acc_system(R=R, Q=Q, count=80)
kf2 = SecondOrderKF(R, Q, dt=1)
est_xs, ps, _, _ = kf2.batch_filter(zs)

nees = NEES(xs, est_xs, ps)
eps = np.mean(nees)

print(f'均值NEES为：{eps:.4f}')
if eps < kf2.dim_x:
    print('通过')
else:
    print('失败')

`NEES`在FilterPy中实现；可使用以下代码进行访问

In [ ]:
from filterpy.stats import NEES

这是一种优秀的滤波器性能度量方法，应该尽可能使用它，特别是在需要在运行滤波器时评估它的性能时。虽然在设计滤波器时，我仍然更喜欢绘制残差图，因为它可以让我更直观地理解发生了什么。

然而，如果您的模拟精度有限，则需要使用另一种方法。

### 似然函数

在统计学中，似然性与概率非常相似，但存在一个微妙的差别，对我们来说非常重要。*概率* 是某事发生的可能性 - 例如，一个公正的骰子在五次掷骰中掷出6的概率是多少？*似然性* 则是相反的问题 - 假设一个骰子在五次掷骰中掷出6三次，那么这个骰子是公正的概率是多少？

我们在**离散贝叶斯**章节中首次讨论了似然函数。在这些滤波器的背景下，似然度量的是当前状态下测量值出现的可能性。

这对我们很重要，因为我们有滤波器输出，我们想知道在高斯噪声和线性行为的假设下，它是否表现最佳。如果可能性很低，我们知道我们的某个假设是错误的。在**自适应滤波**章节中，我们将学习如何利用此信息改进滤波器；在这里我们只学习如何进行测量。

滤波器的残差和系统不确定性定义为

$$\begin{aligned}
\mathbf y &= \mathbf z - \mathbf{H \bar x}\\
\mathbf S &= \mathbf{H\bar{P}H}^\mathsf T + \mathbf R
\end{aligned}
$$

有了这些，我们可以计算可能性函数

$$
\mathcal{L} = \frac{1}{\sqrt{2\pi S}}\exp [-\frac{1}{2}\mathbf y^\mathsf T\mathbf S^{-1}\mathbf y]$$

这看起来很复杂，但请注意指数是高斯方程式。这提示我们可以实现一个：

In [ ]:
from scipy.stats import multivariate_normal
hx = (H @ x).flatten()
S = H @ P @ H.T  + R
likelihood = multivariate_normal.pdf(z.flatten(), mean=hx, cov=S)

在实践中，这种情况会有所不同。数学上处理可能会很困难。通常计算并使用*对数似然度*，它只是似然度的自然对数。这有几个优点。首先，对数是严格增加的，并且在函数的相同点达到最大值。如果您想找到函数的极大值，通常会对其进行导数；找到任意函数的导数可能很困难，但找到$\frac{d}{dx} log(f(x))$却很简单，并且结果和$\frac{d}{dx} f(x)$是相同的。我们在本书中没有使用此属性，但在执行滤波器分析时，这是必不可少的。

当 `update()` 被调用时，我们会为您计算似然和对数似然，它们可以通过数据属性 'log_likelihood' 和 `likelihood` 访问。让我们看看这个例子：我将在预期范围内运行滤波器，然后注入一些远离预期值的测量值：

In [ ]:
R, Q = .05, 0.02
xs, zs = simulate_acc_system(R=R, Q=Q, count=50)
zs[-5:-1] = [100, 200, 200, 200] #坏的测量值！

kf = SecondOrderKF(R, Q, dt=1, P=1)
s = Saver(kf)
kf.batch_filter(zs, saver=s)
plt.plot(s.likelihood);

在开始的几次迭代中，随着滤波器的收敛，似然度会变大。之后，似然度会在一定范围内反弹，直到它到达坏的测量值，此时它会变为零，这表明如果测量值有效，则滤波器很可能不是最优的。

看看对数似然度如何生动地说明滤波器何时会失效。

In [ ]:
plt.plot(s.log_likelihood);

在最后为什么会归零？在看答案之前，先思考一下。滤波器通过将状态移向测量值来适应新的测量值。残差变小，因此状态与残差达成一致。

## 控制输入

在**离散贝叶斯**一章中，我介绍了使用控制信号来提高滤波器性能的概念。我们不再假设物体继续像以前一样移动，而是利用我们对控制输入的了解来预测物体的位置。在**单变量卡尔曼滤波器**一章中，我们利用了同样的想法。卡尔曼滤波器的预测方法如下：

In [ ]:
def predict(pos, movement):
    return (pos[0] + movement[0], pos[1] + movement[1])

在上一章中，我们学习了状态预测的方程式：

$$\bar{\mathbf x} = \mathbf{Fx} + \mathbf{Bu}$$

我们的状态是一个向量，因此我们需要将控制输入表示为向量。 这里$\mathbf{u}$是控制输入，$\mathbf{B}$是一个矩阵，将控制输入转换为$\mathbf x$的变化。让我们考虑一个简单的例子。 假设我们正在控制一个机器人，状态为$x = \begin{bmatrix} x & \dot x\end{bmatrix}$，控制输入为命令速度。 这给了我们一个控制输入

$$\mathbf{u} = \begin{bmatrix}\dot x_\mathtt{cmd}\end{bmatrix}$$

为简单起见，我们将假设机器人可以立即响应对此输入的更改。 这意味着$\Delta t$秒后的新位置和速度将是

$$\begin{aligned}x &= x + \dot x_\mathtt{cmd} \Delta t \\
\dot x &= \dot x_\mathtt{cmd}\end{aligned}$$

我们需要将这组方程表示为$\bar{\mathbf x} = \mathbf{Fx} + \mathbf{Bu}$的形式。

我将使用$\mathbf{Fx}$项提取顶部方程的$x$，并使用$\mathbf{Bu}$项提取其余方程，如下所示：

$$\begin{bmatrix}x\\\dot x\end{bmatrix} = \begin{bmatrix}1 & 0\\0 & 0 \end{bmatrix}\begin{bmatrix}x\\\dot x\end{bmatrix} +
\begin{bmatrix}\Delta t \\ 1\end{bmatrix}\begin{bmatrix}\dot x_\mathtt{cmd}\end{bmatrix}
$$

这是一个简化版；典型的控制输入是转向角的变化和加速度的变化。这会引入非线性，我们将在后面的章节中学习如何处理。

剩下的卡尔曼滤波器将按照正常方式设计。您现在已经看过这个多次了，因此以下是一个示例。

In [ ]:
dt = 1.
R = 3.
kf = Kalman滤波器(dim_x=2, dim_z=1, dim_u = 1)
kf.P *= 10
kf.R *= R
kf.Q = Q_discrete_white_noise(2, dt, 0.1)
kf.F = np.array([[1., 0], [0., 0.]])
kf.B = np.array([[dt], [ 1.]])
kf.H = np.array([[1., 0]])
print(kf.P)

zs = [i + randn()*R for i in range(1, 100)]
xs = []
cmd_velocity = np.array([1.])
for z in zs:
    kf.predict(u=cmd_velocity)
    kf.update(z)
    xs.append(kf.x[0])

In [ ]:
plt.plot(xs, label='KalmanFilter')
plot_measurements(zs)
plt.xlabel('时间')
plt.legend(loc=4)
plt.ylabel('距离');

## 传感器融合

在g-h滤波器章节的早期，我们讨论了为两个尺度设计滤波器的问题，一个准确，一个不准确。我们确定我们应该始终包含不准确滤波器的信息--我们不应该丢弃任何信息。那么考虑一种情况，我们有两个测量系统的传感器。我们应该如何将其纳入我们的卡尔曼滤波器中？

假设我们有一辆火车或者小车在铁路上行驶。它有一个附在车轮上的传感器，用于计算车轮转动的圈数，这可以转换为沿着轨道的距离。接着，假设我们有一个类似 GPS 的传感器，我称之为“位置传感器”，安装在火车上报告位置。我将在下一节中解释为什么不直接使用 GPS。因此，我们有两个测量值，都报告了轨道上的位置。进一步假设车轮传感器的精度为1米，位置传感器的精度为10米。我们如何将这两个测量值组合成一个滤波器？这可能看起来很牵强，但飞机使用传感器融合技术来融合来自 GPS、INS、多普勒雷达、VOR、空速指示器等传感器的测量值。

惯性系统的卡尔曼滤波器非常困难，但是融合两个或多个提供同一状态变量（如位置）测量的传感器数据相当容易。相关的矩阵是测量矩阵 $\mathbf H$。请记住，该矩阵告诉我们如何从卡尔曼滤波器的状态 $\mathbf x$ 转换为测量 $\mathbf z$。假设我们决定我们的卡尔曼滤波器状态应包含火车的位置和速度，因此

$$ \mathbf x = \begin{bmatrix}x \\ \dot x\end{bmatrix}$$

我们有两个位置测量值，因此我们将定义测量向量为从轮子和位置传感器中得到的测量值向量。

$$ \mathbf z = \begin{bmatrix}z_{wheel} \\ z_{ps}\end{bmatrix}$$

因此，我们必须设计矩阵 $\mathbf H$ 将 $\mathbf x$ 转换为 $\mathbf z$。它们都是位置，因此转换不过是乘以1：

$$ \begin{bmatrix}z_{wheel} \\ z_{ps}\end{bmatrix} = \begin{bmatrix}1 &0 \\ 1& 0\end{bmatrix} \begin{bmatrix}x \\ \dot x\end{bmatrix}$$

为了更清晰，假设轮子报告的不是位置而是轮子的旋转次数，其中1个旋转产生2米的行程。在这种情况下，我们会写成

$$ \begin{bmatrix}z_{rot} \\ z_{ps}\end{bmatrix} = \begin{bmatrix}0.5 &0 \\ 1& 0\end{bmatrix} \begin{bmatrix}x \\ \dot x\end{bmatrix}$$

现在我们要设计测量噪声矩阵$\mathbf R$。假设位置的测量方差是轮子方差的两倍，轮子的标准差为1.5米。这给了我们

$$
\begin{aligned}
\sigma_{wheel} &=  1.5\\
\sigma^2_{wheel} &= 2.25 \\ 
\sigma_{ps} &= 1.5*2 = 3 \\
\sigma^2_{ps} &= 9.
\end{aligned}
$$

这就是我们的卡尔曼滤波器设计。我们需要为 $\mathbf Q$ 进行设计，但这与是否进行传感器融合无关，因此我将选择一些任意值。

现在我们来运行一下这个设计的模拟。我将假设速度为 10 米/秒，更新率为 0.1 秒。

In [ ]:
from numpy import array, asarray
import numpy.random as random

def fusion_test(wheel_sigma, ps_sigma, do_plot=True):
    dt = 0.1
    kf = KalmanFilter(dim_x=2, dim_z=2)

    kf.F = array([[1., dt], [0., 1.]])
    kf.H = array([[1., 0.], [1., 0.]])
    kf.x = array([[0.], [1.]])
    kf.Q *= array([[(dt**3)/3, (dt**2)/2],
                   [(dt**2)/2,  dt      ]]) * 0.02
    kf.P *= 100
    kf.R[0, 0] = wheel_sigma**2
    kf.R[1, 1] = ps_sigma**2 
    s = Saver(kf)

In [ ]:
random.seed(1123)
    for i in range(1, 100):
        m0 = i + randn()*wheel_sigma
        m1 = i + randn()*ps_sigma
        kf.predict()
        kf.update(array([[m0], [m1]]))
        s.save()
    s.to_array()
    print(f'融合标准差: {np.std(s.y[:, 0]):.3f}')
    if do_plot:
        ts = np.arange(0.1, 10, .1)
        plot_measurements(ts, s.z[:, 0], label='轮子')
        plt.plot(ts, s.z[:, 1], ls='--', label='位置传感器')
        plot_filter(ts, s.x[:, 0], label='KalmanFilter')
        plt.legend(loc=4)
        plt.ylim(0, 100)
        set_labels(x='时间 (秒)', y='米')
        
融合测试(1.5, 3.0)

我们可以看到卡尔曼滤波器的结果以蓝色显示。

可能有些难以直观理解前面的例子。让我们看看一个不同的问题。假设我们在二维空间中跟踪一个物体，并且有两个不同位置的雷达系统。每个雷达系统都给我们目标的距离和方位角。每个数据的读数如何影响结果？

这是一个非线性问题，因为我们需要使用三角函数从距离和方位角计算坐标，并且我们还没有学会如何使用卡尔曼滤波器解决非线性问题。因此，对于这个问题，忽略我使用的代码，只专注于代码输出的图表。我们将在后续章节中重新访问这个问题并学习如何编写此代码。

我将目标定位在（100，100）处。第一个雷达将位于（50，50），第二个雷达将位于（150，50）。这将导致第一个雷达测量出45度的方位角，第二个雷达报告135度。

首先，我将创建卡尔曼滤波器，然后绘制其初始协方差矩阵。我使用的是**无迹卡尔曼滤波器**，这在后面的章节中会讲到。

In [ ]:
from kf_book.kf_design_internal import sensor_fusion_kf

kf = sensor_fusion_kf()
x0, p0 = kf.x.copy(), kf.P.copy()
plot_covariance_ellipse(x0, p0, fc='y', ec=None, alpha=0.6)

我们对x和y的位置同样不确定，因此协方差是圆形的。

现在我们将使用第一个雷达的读数更新卡尔曼滤波器。我将把方位误差的标准差设置为0.5$^\circ$，距离误差的标准差设置为3。

In [ ]:
from math import radians
from kf_book.kf_design_internal import sensor_fusion_kf, set_radar_pos

# 设置雷达方位和距离误差
kf.R[0, 0] = radians (.5)**2
kf.R[1, 1] = 3.**2

# 从第一个雷达站计算位置和协方差
set_radar_pos((50, 50))
dist = (50**2 + 50**2) ** 0.5
kf.predict()
kf.update([radians(45), dist])

# 绘制结果
x1, p1 = kf.x.copy(), kf.P.copy()

plot_covariance_ellipse(x0, p0, fc='y', ec=None, alpha=0.6)
plot_covariance_ellipse(x1, p1, fc='g', ec='k', alpha=0.6)

plt.scatter([100], [100], c='y', label='初始')
plt.scatter([100], [100], c='g', label='第一个站')
plt.legend(scatterpoints=1, markerscale=3)
plt.plot([92, 100], [92, 100], c='g', lw=2, ls='--');

我们可以看到误差对问题几何形状的影响。雷达站位于目标的左下方。方位角测量非常准确，$\sigma=0.5^\circ$，但距离误差不准确，$\sigma=3$。我们用虚线绿色线显示雷达读数。我们可以通过协方差椭圆的形状轻松看出方位角准确和距离不准确的影响。

现在我们可以加入第二个雷达站的测量结果。第二个雷达站在目标物体的下方和右侧位置为(150,50)。在继续之前，请思考在加入这个新的读数时，协方差矩阵会如何变化。

In [ ]:
# 计算第二个雷达站的位置和协方差
set_radar_pos((150, 50))
kf.predict()
kf.update([radians(135), dist])

plot_covariance_ellipse(x0, p0, fc='y', ec='k', alpha=0.6)
plot_covariance_ellipse(x1, p1, fc='g', ec='k', alpha=0.6)
plot_covariance_ellipse(kf.x, kf.P, fc='b', ec='k', alpha=0.6)

plt.scatter([100], [100], c='y', label='初始值')
plt.scatter([100], [100], c='g', label='第一雷达站')
plt.scatter([100], [100], c='b', label='第二雷达站')
plt.legend(scatterpoints=1, markerscale=3)
plt.plot([92, 100], [92, 100], c='g', lw=2, ls='--')
plt.plot([108, 100], [92, 100], c='b', lw=2, ls='--');

我们可以看到第二个雷达测量是如何改变协方差的。目标的角度与第一个雷达站点垂直，因此方位和距离误差的影响被交换了。因此，协方差矩阵的角度会转换以匹配到第二个站点的方向。需要注意的是，方向不仅仅改变了；协方差矩阵的大小也变得小得多。

协方差将始终包含所有可用信息，包括问题几何形状的影响。这种表述使得看清发生了什么特别容易，但是如果一个传感器提供位置，另一个传感器提供速度，或者如果两个传感器提供位置测量，同样的事情也会发生。

在继续之前，有一件重要的事情要注意：传感器融合是一个广泛的主题，而我的覆盖面过于简单，甚至会误导。例如，GPS使用迭代最小二乘法来确定卫星的一组伪距读数的位置，而不使用卡尔曼滤波器。我在支持笔记本[**迭代最小二乘用于传感器融合**](http://nbviewer.ipython.org/urls/raw.github.com/rlabbe/Kalman-and-Bayesian-Filters-in-Python/master/Supporting_Notebooks/Iterative-Least-Squares-for-Sensor-Fusion.ipynb)中介绍了这个主题。

这是GPS接收器中通常但并不是独有的计算方法。如果您是一个业余爱好者，我的介绍可以让您入门。商业级别的滤波器需要非常谨慎地设计融合过程。这是几本书的主题，您需要找到一本涵盖您领域的书籍来进一步学习。

### 练习：您能过滤GPS输出吗？

在上面的部分中，我让你将卡尔曼滤波器应用于“类GPS”的传感器。你可以将卡尔曼滤波器应用于商用卡尔曼滤波器的输出吗？换句话说，你的滤波器的输出会比GPS的输出更好、更差还是相等？

#### 解决方案

商用GPS中内置了卡尔曼滤波器，它们的输出是由该滤波器创建的滤波估计。因此，假设你有一个由位置和位置误差组成的GPS输出的稳定流。你不能将这两个数据传递到你自己的滤波器中吗？

那么，这个数据流的特性是什么？更重要的是，卡尔曼滤波器的输入的基本要求是什么？

Kalman滤波器的输入必须是*高斯*和*时间无关的*。这是因为我们施加了马尔科夫属性的要求：当前状态仅取决于先前状态和当前输入。这使得滤波器的递归形式成为可能。GPS的输出是*时间相关的*，因为滤波器基于所有先前测量的递归估计来估计当前状态。因此，信号不是白色的，它不是时间无关的，如果你将这些数据传入Kalman滤波器，你就违反了滤波器的数学要求。因此，答案是否定的，你不能通过在商用GPS的输出上运行KF来获得更好的估计。

另一个考虑卡尔曼滤波器的角度是，它们在最小二乘意义下是最优的。没有办法将最优解通过任何滤波器进行处理，得到一个更优的答案，因为这在逻辑上是不可能的。最好的情况是，信号不会改变，此时仍然是最优的，或者信号会发生变化，从而不再是最优的。

这是业余爱好者在尝试集成GPS、IMU和其他现成传感器时面临的难题。

让我们看看效果。商用 GPS 报告位置和估计的误差范围。估计的误差只来自卡尔曼滤波器的 $\mathbf P$ 矩阵。因此，让我们过滤一些噪声数据，将滤波后的输出作为滤波器的新噪声输入，并查看结果。换句话说，$\mathbf x$ 将提供 $\mathbf z$ 输入，$\mathbf P$ 将提供测量协方差 $\mathbf R$。为了夸大效果使其更加明显，我将绘制进行一次滤波后的效果，然后再进行第二次滤波，绘制其效果。第二次迭代没有任何“意义”（没有人会尝试这样做），这只是帮助我阐述一个观点。首先，是代码和图表。

In [ ]:
np.random.seed(124)
R = 5.
xs, zs = simulate_acc_system(R=R, Q=Q, count=30)

kf0 = SecondOrderKF(R, Q, dt=1)
kf1 = SecondOrderKF(R, Q, dt=1)
kf2 = SecondOrderKF(R, Q, dt=1)

# 过滤测量值
fxs0, ps0, _, _ = kf0.batch_filter(zs)

# 使用状态作为输入进行两次滤波
fxs1，ps1，_，_ = kf1.batch_filter（fxs0 [：，0]）
fxs2，_，_，_ = kf2.batch_filter（fxs1 [：，0]）

plot_kf_output（xs，fxs0，zs，'KF'，False）
plot_kf_output（xs，fxs1，zs，'1次迭代'，False）
plot_kf_output（xs，fxs2，zs，'2次迭代'，False）
R，Q

In [ ]:

我们看到重新处理后的信号的滤波输出更加平滑，但也偏离了轨道。发生了什么？回想一下，KalmanFilter要求信号不具有时间相关性。但是，KalmanFilter的输出是具有时间相关性的，因为它将所有先前的测量值合并到其对此时间段的估计中。因此看看最后一个图表，即2次迭代的图表。测量值从几个高于轨道的峰值开始。滤波器“记住”了这一点（这是模糊的术语，但我试图避免使用数学），并开始计算物体高于轨道。稍后，在大约13秒时，我们有一段时间内所有测量值都恰好低于轨道。这也被合并到滤波器的记忆中，并使迭代输出偏离轨道很远。

现在让我们用另一种方式来看待这个问题。迭代输出*不是*使用$\mathbf z$作为测量值，而是使用上一个KalmanFilter估计的输出。因此，我将绘制滤波器的输出与上一个滤波器的输出相对比。

```python
plot_kf_output(xs, fxs0, zs, title='KF', aspect_equal=False)
plot_kf_output(xs, fxs1, fxs0[:, 0], '1 iteration', False)
plot_kf_output(xs, fxs2, fxs1[:, 0], '2 iterations', False)

我希望这种方法的问题现在已经显而易见了。在底部的图表中，我们可以看到KF正在跟踪上一个滤波器的不完美估计，并且由于上一个测量值的记忆被合并到信号中，导致信号中也存在延迟。

### 练习：证明位置传感器可以改善滤波器

设计一种方法，证明将位置传感器和轮子测量值融合产生的结果优于仅使用轮子测量值。

#### 解决方案1

强制卡尔曼滤波器忽略位置传感器的测量，将位置传感器的测量噪声设置为接近无穷大的值。重新运行滤波器并观察残差的标准偏差。

In [ ]:
fusion_test(1.5, 3.0, do_plot=False)
fusion_test(1.5, 1.e40, do_plot=False)

在这里，我们可以看到，当位置传感器测量几乎被忽略时，滤波器的误差大于使用位置传感器时的误差。

#### 解决方案2

这需要更多的工作，但我们可以编写一个仅采用一个测量的卡尔曼滤波器。

In [ ]:
dt = 0.1
wheel_sigma = 1.5
kf = KalmanFilter(dim_x=2, dim_z=1)
kf.F = array([[1., dt], [0., 1.]])
kf.H = array([[1., 0.]])
kf.x = array([[0.], [1.]])
kf.Q *= 0.01
kf.P *= 100
kf.R[0, 0] = wheel_sigma**2

random.seed(1123)
nom = range(1, 100)
zs = np.array([i + randn()*wheel_sigma for i in nom])
xs, _, _, _ = kf.batch_filter(zs)
ts = np.arange(0.1, 10, .1)

res = nom - xs[:, 0, 0]
print(f'std: {np.std(res):.3f}')

In [ ]:
plot_filter(ts, xs[:, 0], label='KalmanFilter')
plot_measurements(ts, zs, label='轮子')
set_labels(x='时间(秒)', y='米')
plt.legend(loc=4);

在这次运行中，我得到了标准差为0.523，而融合测量值的标准差为0.391。

## 非定常过程

到目前为止，我们假设卡尔曼滤波器中的各种矩阵是*定常的*——即随时间不变。例如，在机器人跟踪部分，我们假设$\Delta t=1.0$秒，并设计状态转移矩阵为

$$
\mathbf F = \begin{bmatrix}1& \Delta t& 0& 0\\0& 1& 0& 0\\0& 0& 1& \Delta t\\ 0& 0& 0& 1\end{bmatrix} = \begin{bmatrix}1& 1& 0& 0\\0& 1& 0& 0\\0& 0& 1& 1\\ 0& 0& 0& 1\end{bmatrix}$$

但是，如果我们的数据速率以某种不可预测的方式发生变化呢？或者，如果我们有两个传感器，每个传感器以不同的速率运行？测量误差发生变化怎么办？

处理这很容易；您只需更改卡尔曼滤波器矩阵以反映当前情况。让我们回到我们的狗跟踪问题，并假设数据输入有些间歇性。对于这个问题，我们设计了

$$\begin{aligned}
\mathbf{\bar x} &= {\begin{bmatrix}x\\\dot x\end{bmatrix}}^- \\
\mathbf F &= \begin{bmatrix}1&\Delta t  \\ 0&1\end{bmatrix} 
\end{aligned}$$

并在初始化期间将卡尔曼滤波器变量`F`设置如下：

In [ ]:
dt = 0.1
kf.F = np.array([[1, dt],
                 [0, 1]])

如果每个测量的$\Delta t$都不同，我们该如何处理？这很容易-只需修改相关矩阵。在这种情况下，`F`是变量，因此我们需要在更新/预测循环内更新它。 `Q`也取决于时间，因此每个循环都必须进行分配。以下是我们可能编写此代码的示例：

In [ ]:
kf = KalmanFilter(dim_x=2, dim_z=1)
kf.x = array([0., 1.])
kf.H = array([[1, 0]])
kf.P = np.eye(2) * 50
kf.R = np.eye(1)
q_var = 0.02

# measurement tuple: (value, time)
zs = [(1., 1.),  (2., 1.1), (3., 0.9), (4.1, 1.23), (5.01, 0.97)]
for z, dt in zs:
    kf.F = array([[1, dt],
                  [0, 1]])
    kf.Q = Q_discrete_white_noise(dim=2, dt=dt, var=q_var)
    kf.predict()
    kf.update(z)
    print(kf.x)

### 传感器融合：不同的数据速率

很少有两个不同传感器类以相同的速率输出数据。假设位置传感器以 3 Hz 的速率产生更新，轮子以 7 Hz 的速率更新。进一步假设时间不是很精确——存在一些抖动，使得测量可能会在预测时间之前或之后发生一点。让我通过让轮子提供速度估计而不是位置估计来进一步复杂化情况。

我们可以通过等待来自任一传感器的数据包来实现此目的。当我们收到它时，我们确定自上次更新以来经过的时间量。然后我们需要修改受影响的矩阵。 $\mathbf F$和 $\mathbf Q$ 都包含一个时间项 $\Delta t$，因此我们将需要在每次创新时调整这些内容。

测量每次更改，因此我们将不得不修改 $\mathbf H$ 和 $\mathbf R$。位置传感器更改 $\mathbf x$ 的位置元素，因此我们分配：

$$\begin{aligned}
\mathbf H &= \begin{bmatrix}1 &0\end{bmatrix} \\
\mathbf R &= \sigma_{ps}^2
\end{aligned}$$

轮传感器更改 $\mathbf x$ 的速度元素，因此我们分配：

$$\begin{aligned}
\mathbf H &= \begin{bmatrix}0 &1\end{bmatrix} \\
\mathbf R &= \sigma_{wheel}^2
\end{aligned}$$

In [ ]:
def gen_sensor_data(t, ps_std, wheel_std):
    # 生成模拟传感器数据
    pos_data, vel_data = [], []
    dt = 0.
    for i in range(t*3):
        dt += 1/3.
        t_i = dt + randn() * .01 # 时间抖动
        pos_data.append([t_i, t_i + randn()*ps_std])

    dt = 0.    
    for i in range(t*7):
        dt += 1/7.
        t_i = dt + randn() * .006 # 时间抖动
        vel_data.append([t_i, 1. + randn()*wheel_std])
    return pos_data, vel_data


def plot_fusion(xs, ts, zs_ps, zs_wheel):
    xs = np.array(xs)
    plt.subplot(211)
    plt.plot(zs_ps[:, 0], zs_ps[:, 1], ls='--', label='位置传感器')
    plot_filter(xs=ts, ys=xs[:, 0], label='KalmanFilter')
    set_labels(title='位置', y='米',)

    plt.subplot(212)
    plot_measurements(zs_wheel[:, 0], zs_wheel[:, 1], label='车轮')
    plot_filter(xs=ts, ys=xs[:, 1], label='KalmanFilter')
    set_labels('速度', '时间 (秒)', '米/秒')

In [ ]:
def fusion_test(pos_data, vel_data, wheel_std, ps_std):
    kf = KalmanFilter(dim_x=2, dim_z=1)
    kf.F = array([[1., 1.], [0., 1.]])
    kf.H = array([[1., 0.], [1., 0.]])
    kf.x = array([[0.], [1.]])
    kf.P *= 100

    xs, ts = [],  []

    # 用于绘制图表的数据
    zs_wheel = np.array(vel_data)
    zs_ps = np.array(pos_data)

    last_t = 0
    while len(pos_data) > 0 and len(vel_data) > 0:
        if pos_data[0][0] < vel_data[0][0]:
            t, z = pos_data.pop(0)
            dt = t - last_t
            last_t = t

            kf.H = np.array([[1., 0.]])
            kf.R[0,0] = ps_std**2
        else:
            t, z = vel_data.pop(0)
            dt = t - last_t
            last_t = t

            kf.H = np.array([[0., 1.]])
            kf.R[0,0] = wheel_std**2

        kf.F[0,1] = dt
        kf.Q = Q_discrete_white_noise(2, dt=dt, var=.02)
        kf.predict()
        kf.update(np.array([z]))

In [ ]:
xs.append(kf.x.T[0])
        ts.append(t)
    plot_fusion(xs, ts, zs_ps, zs_wheel)

random.seed(1123)
pos_data, vel_data = gen_sensor_data(25, 1.5, 3.0)
fusion_test(pos_data, vel_data, 1.5, 3.0);

## 跟踪一个球

现在让我们将注意力转向一个受限的物体物理情况下跟踪的情况。在真空中投掷的球必须遵守牛顿定律。在恒定的重力场中，它将沿着一条抛物线运动。我假设您已熟悉了以下公式的推导：

$$
\begin{aligned}
y &= \frac{g}{2}t^2 + v_{y0} t + y_0 \\
x &= v_{x0} t + x_0
\end{aligned}
$$

其中$g$是重力常数，$t$是时间，$v_{x0}$和$v_{y0}$是x和y平面上的初始速度。如果以$\theta$角度将球以初速度$v$投掷，则可以计算$v_{x0}$和$v_{y0}$为

$$
\begin{aligned}
v_{x0} = v \cos{\theta} \\
v_{y0} = v \sin{\theta}
\end{aligned}
$$

因为我们没有真实data，所以我们将从编写一个球的模拟器开始。像往常一样，我们添加了一个与时间无关的noise项，以便可以模拟有noise的传感器。

```python
from math import radians, sin, cos
import math

def rk4(y, x, dx, f):
    """计算dy/dx的四阶龙格库塔。
    y是y的初始值
    x是x的初始值
    dx是x的差异(例如时间步长)
    f是一个可调用函数(y, x)，您提供它来计算指定值的dy/dx。
    """

    k1 = dx * f(y, x)
    k2 = dx * f(y + 0.5*k1, x + 0.5*dx)
    k3 = dx * f(y + 0.5*k2, x + 0.5*dx)
    k4 = dx * f(y + k3, x + dx)

    return y + (k1 + 2*k2 + 2*k3 + k4) / 6.

def fx(x,t):
    return fx.vel

def fy(y,t):
    return fy.vel - 9.8*t

```python
class BallTrajectory2D(object):
    def __init__(self, x0, y0, velocity, 
                 theta_deg=0., 
                 g=9.8, 
                 noise=[0.0, 0.0]):
        self.x = x0
        self.y = y0
        self.t = 0        
        theta = math.radians(theta_deg)
        fx.vel = math.cos(theta) * velocity
        fy.vel = math.sin(theta) * velocity        
        self.g = g
        self.noise = noise
        
        
    def step(self, dt):
        self.x = rk4(self.x, self.t, dt, fx)
        self.y = rk4(self.y, self.t, dt, fy)
        self.t += dt 
        return (self.x + randn()*self.noise[0], 
                self.y + randn()*self.noise[1])

要创建一个以 (0, 15) 为起点，速度为 100m/s，角度为 60° 的轨迹，我们可以写成：

In [ ]:
traj = BallTrajectory2D(x0=0, y0=15, velocity=100, theta_deg=60)

然后为每个时间步骤调用 `traj.step(t)`。让我们测试一下。

In [ ]:
def test_ball_vacuum(noise):
    y = 15
    x = 0
    ball = BallTrajectory2D(x0=x, y0=y, 
                            theta_deg=60., velocity=100., 
                            noise=noise)
    t = 0
    dt = 0.25
    while y >= 0:
        x, y = ball.step(dt)
        t += dt
        if y >= 0:
            plt.scatter(x, y, color='r', marker='.', s=75, alpha=0.5)
         
    plt.axis('equal');
    
#test_ball_vacuum([0, 0]) # 绘制理想球的位置
test_ball_vacuum([1, 1]) # 绘制有噪音的球的位置 


看起来很合理，那么我们继续（读者的练习：更加健壮地验证此模拟）。

### 选择状态变量

我们可能会想使用与跟踪狗时相同的状态变量。但是，这样做是不行的。回想一下，Kalman滤波器的状态转移必须写成$\mathbf{\bar x} = \mathbf{Fx} + \mathbf{Bu}$，这意味着我们必须从先前的状态计算当前状态。我们的假设是球在真空中运行，因此x方向上的速度是恒定的，y方向上的加速度仅由重力常数$g$决定。我们可以使用以$\Delta t$为单位的欧拉方法离散化牛顿方程，如下所示：

$$\begin{aligned}
x_t &=  x_{t-1} + v_{x(t-1)} {\Delta t} \\
v_{xt} &= v_{x(t-1)} \\
y_t &= y_{t-1} + v_{y(t-1)} {\Delta t} \\
v_{yt} &= -g {\Delta t} + v_{y(t-1)} \\
\end{aligned}
$$

> **侧边栏**: *欧拉方法通过假设时间$t$处的斜率（导数）是恒定的，逐步积分微分方程。在这种情况下，位置的导数是速度。在每个时间步长$\Delta t$处，我们假设一个恒定的速度，计算新的位置，然后更新下一个时间步长的速度。虽然有更准确的方法，例如Runge-Kutta可供使用，但因为我们在每个步骤中使用测量更新状态，欧拉方法非常准确。如果您需要使用Runge-Kutta，您将不得不编写自己的`predict()`函数，该函数计算$\mathbf x$的状态转换，然后使用正常的卡尔曼滤波器方程$\mathbf{\bar P}=\mathbf{FPF}^\mathsf T + \mathbf Q$来更新协方差矩阵。*

这意味着我们需要将$y$的加速度纳入卡尔曼滤波器，但不需要将$x$的加速度纳入。这表明以下状态变量。

$$
\mathbf x = 
\begin{bmatrix}
x & \dot x & y & \dot y & \ddot{y}
\end{bmatrix}^\mathsf T
$$

然而，加速度是由重力引起的，而重力是一个常量。我们可以将重力视为实际情况——一个控制输入，而不是让卡尔曼滤波器去跟踪一个常量。换句话说，重力是一种以已知方式改变系统行为的力，并且在整个球的飞行过程中都会施加。

状态预测方程为 $\mathbf{\bar x} = \mathbf{Fx} + \mathbf{Bu}$。其中，$\mathbf{Fx}$ 是我们用来模拟球的位置和速度的状态转移函数。向量 $\mathbf{u}$ 允许您将控制输入输入到滤波器中。对于汽车，控制输入将包括加速器和制动器的踏板程度、方向盘的位置等。对于我们的球，控制输入将是重力。矩阵 $\mathbf{B}$ 模拟控制输入对系统行为的影响。同样，对于汽车，$\mathbf{B}$ 将制动器和加速器的输入转换为速度变化，并将方向盘的输入转换为不同的位置和朝向。对于我们的球跟踪问题，它将计算由于重力而产生的速度变化。我们很快就会详细介绍这一点。现在，我们将设计状态变量为：

$$
\mathbf x = 
\begin{bmatrix}x & \dot x & y & \dot y 
\end{bmatrix}^\mathsf T
$$

### 设计状态转移函数

我们的下一步是设计状态转移函数。回想一下，状态转移函数被实现为一个矩阵 $\mathbf F$，我们将其与系统的先前状态相乘，以获得下一个状态或先验 $\bar{\mathbf x} = \mathbf{Fx}$。

我不会详细阐述这一点，因为它与我们在上一章中进行的 1-D 情况非常相似。我们位置和速度的状态方程式将为：

$$
\begin{aligned}
\bar x &= (1*x) + (\Delta t * v_x) + (0*y) + (0 * v_y) \\
\bar v_x &= (0*x) +  (1*v_x) + (0*y) + (0 * v_y) \\
\bar y &= (0*x) + (0* v_x)         + (1*y) + (\Delta t * v_y)   \\
\bar v_y &= (0*x) +  (0*v_x) + (0*y) + (1*v_y) 
\end{aligned}
$$

请注意，所有术语都不包括 $g$，即重力常数。正如我在前面的函数中所解释的那样，我们将使用卡尔曼滤波器的控制输入来考虑重力因素。

我们按矩阵形式写出如下的表达式：

$$
\mathbf F = \begin{bmatrix}
1 & \Delta t & 0 & 0 \\
0 & 1 & 0 & 0 \\
0 & 0 & 1 & \Delta t \\
0 & 0 & 0 & 1
\end{bmatrix}
$$

### 设计控制输入函数

我们将使用控制输入来考虑重力的作用。将 $\mathbf{Bu}$ 添加到 $\mathbf{\bar x}$ 中，以考虑由于重力而引起的 $\mathbf{\bar x}$ 的变化量。我们可以说 $\mathbf{Bu}$ 包含 $\begin{bmatrix}\Delta x_g & \Delta \dot{x_g} & \Delta y_g & \Delta \dot{y_g}\end{bmatrix}^\mathsf T$。

如果我们查看离散化的方程式，我们会发现重力只会影响 $y$ 轴的速度。

$$\begin{aligned}
x_t &=  x_{t-1} + v_{x(t-1)} {\Delta t} \\
v_{xt} &= vx_{t-1}
\\
y_t &= y_{t-1} + v_{y(t-1)} {\Delta t}\\
v_{yt} &= -g {\Delta t} + v_{y(t-1)} \\
\end{aligned}
$$

因此，我们希望$\mathbf{Bu}$的乘积等于$\begin{bmatrix}0 & 0 & 0 & -g \Delta t \end{bmatrix}^\mathsf T$。从某种意义上说，我们定义$\mathbf{B}$和$\mathbf{u}$的方式是任意的，只要它们相乘得到这个结果。例如，我们可以定义$\mathbf{B}=1$，$\mathbf{u} = \begin{bmatrix}0 & 0 & 0 & -g \Delta t \end{bmatrix}^\mathsf T$。但这与我们对$\mathbf{B}$和$\mathbf{u}$的定义不太符合，其中$\mathbf{u}$是控制输入，$\mathbf{B}$是控制函数。控制输入为$-g$，用于y轴的速度。因此，这是一种可能的定义。

$$\mathbf{B} = \begin{bmatrix}0&0&0&0 \\ 0&0&0&0 \\0&0&0&0 \\0&0&0&\Delta t\end{bmatrix}, \mathbf{u} = \begin{bmatrix}0\\0\\0\\-g\end{bmatrix}$$

对我来说，这似乎有点过度，我建议$\mathbf{u}$包含两个维度$x$和$y$的控制输入，这表明

$$\mathbf{B} = \begin{bmatrix}0&0 \\ 0&0 \\0&0 \\0&\Delta t\end{bmatrix}, \mathbf{u} = \begin{bmatrix}0\\-g\end{bmatrix}$$。

你可能更喜欢仅提供实际存在的控制输入，而对于$x$没有控制输入，因此我们得到

$$\mathbf{B} = \begin{bmatrix}0 \\ 0 \\0\\ \Delta t\end{bmatrix}, \mathbf{u} = \begin{bmatrix}-g\end{bmatrix}$$。

我看到有人使用

$$\mathbf{B} = \begin{bmatrix}0&0&0&0 \\ 0&0&0&0 \\0&0&0&0 \\0&0&0&1\end{bmatrix}, \mathbf{u} = \begin{bmatrix}0\\0\\0\\-g \Delta t\end{bmatrix}$$

虽然这样会产生正确的结果，但我不愿意将时间放入$\mathbf{u}$中，因为时间不是控制输入，它是我们用来将控制输入转换为状态变化的东西，而这是$\mathbf{B}$的工作。

### 设计测量函数

测量函数定义了我们如何使用方程$\mathbf z = \mathbf{Hx}$从状态变量到测量结果。我们将假设我们有一个传感器，可以为我们提供球的(x,y)位置，但不能测量速度或加速度。因此，我们的函数必须是：

$$
\begin{bmatrix}z_x \\ z_y \end{bmatrix}= 
\begin{bmatrix}
1 & 0 & 0 & 0 \\
0 & 0 & 1 & 0
\end{bmatrix} 
\begin{bmatrix}
x \\
\dot x \\
y \\
\dot y \end{bmatrix}$$

其中

$$\mathbf H = \begin{bmatrix}
1 & 0 & 0 & 0 \\
0 & 0 & 1 & 0
\end{bmatrix}$$

### 设计测量噪声矩阵

与机器人一样，我们将假设误差在$x$和$y$中是独立的。在这种情况下，我们将首先假设x和y的测量误差为0.5平方米。因此，

$$\mathbf R = \begin{bmatrix}0.5&0\\0&0.5\end{bmatrix}$$

### 设计过程噪声矩阵

我们假设一个在真空中运动的球，所以不应该有过程噪声。我们有4个状态变量，所以需要一个 $4{\times}4$ 协方差矩阵：

$$\mathbf Q = \begin{bmatrix}0&0&0&0\\0&0&0&0\\0&0&0&0\\0&0&0&0\end{bmatrix}$$

### 设计初始条件

我们在测试状态转移函数时已经执行了此步骤。回想一下，我们使用三角函数计算了 $x$ 和 $y$ 的初始速度，并使用以下代码设置了 $\mathbf x$ 的值：

In [ ]:
omega = radians(omega)
vx = cos(omega) * v0
vy = sin(omega) * v0

f1.x = np.array([[x, vx, y, vy]]).T

完成所有步骤后，我们准备实现过滤器并进行测试。首先是实现：

In [ ]:
from math import sin, cos, radians

def ball_kf(x, y, omega, v0, dt, r=0.5, q=0.):
    kf = KalmanFilter(dim_x=4, dim_z=2, dim_u=1)

kf.F = np.array([[1., dt, 0., 0.],   # x   = x0 + dx*dt
                     [0., 1., 0., 0.],   # dx  = dx0
                     [0., 0., 1., dt],   # y   = y0 + dy*dt
                     [0., 0., 0., 1.]])  # dy  = dy0

In [ ]:
kf.H = np.array([[1., 0., 0., 0.],
                 [0., 0., 1., 0.]])

kf.B = np.array([[0., 0., 0., dt]]).T
kf.R *= r
kf.Q *= q

omega = radians(omega)
vx = cos(omega) * v0
vy = sin(omega) * v0
kf.x = np.array([[x, vx, y, vy]]).T
return kf
```

现在我们将使用球体模拟类为球体生成测量值来测试滤波器。

In [ ]:
def track_ball_vacuum(dt):
    global kf
    x, y = 0., 1.
    theta = 35.  # 发射角度
    v0 = 80.
    g = np.array([[-9.8]])  # 重力常数
    ball = BallTrajectory2D(x0=x, y0=y, theta_deg=theta, velocity=v0, 
                            noise=[.2, .2])
    kf = ball_kf(x, y, theta, v0, dt)


t = 0
    xs, ys = [], []
    while kf.x[2] > 0:
        t += dt
        x, y = ball.step(dt)
        z = np.array([[x, y]]).T

In [ ]:
kf.update(z)
xs.append(kf.x[0])
ys.append(kf.x[2])    
kf.predict(u=g)     
p1 = plt.scatter(x, y, color='r', marker='.', s=75, alpha=0.5)
    p2, = plt.plot(xs, ys, lw=2)
    plt.legend([p2, p1], ['KalmanFilter', '测量值'],
       scatterpoints=1)

track_ball_vacuum(dt=1./10)

In [ ]:

我们看到，KalmanFilter可以很好地跟踪球。然而，正如已经解释的那样，这是一个微不足道的例子，因为我们没有过程noise。我们可以用任意精度在真空中预测轨迹；在这个例子中使用KalmanFilter是一种不必要的复杂化。最小二乘曲线拟合将给出相同的结果。

## 跟踪空中球

为了解决这个问题，我们假设我们正在追踪一颗通过地球大气层运行的球。球的轨迹受到风、阻力和旋转的影响。我们假设我们的传感器是一台摄像头；我们不会实现的代码将执行某种类型的图像处理来检测球的位置。在计算机视觉中，这通常被称为*斑点检测*。然而，图像处理代码并不完美；在任何给定的帧中，可能会检测不到斑点，或者检测到不对应于球的虚假斑点。最后，我们不会假设我们知道球的起始位置、角度或旋转；跟踪代码将必须根据提供的测量结果启动跟踪。我们在这里做出的主要简化是一个二维世界；我们假设球总是垂直于摄像头传感器平面行驶。因为我们还没有讨论如何...

可能会从仅提供2Ddata的相机中提取3D信息。

### 实现空气阻力

我们的第一步是实现空气中球运动的数学模型。有多种处理方法可供选择。一个强健的解决方案考虑到问题，例如球的粗糙度（这会根据速度非线性地影响阻力），Magnus效应（旋转导致球的一侧相对于空气具有更高的速度，相对于相反的侧面，因此阻力系数在相反的侧面上有所不同），升力、湿度、空气密度等等。我假设读者对于球体物理学的细节不感兴趣，因此将限制这种处理对于非旋转棒球的空气阻力的影响。我将使用Nicholas Giordano和Hisao Nakanishi在《计算物理学》[1997]中开发的数学方法。这种处理方法没有考虑到所有因素。最详细的处理方法是Alan Nathan在他的网站http://baseball.physics.illinois.edu/index.html上提供的。我在我的计算机视觉工作中使用他的数学方法，但我不想被其他因素分心。

复杂模型。

**重要提示**：在我继续之前，让我指出，您不必理解下一段物理知识即可继续使用KalmanFilter。我的目标是在现实世界中创建一个相当准确的棒球行为，以便我们可以测试我们的KalmanFilter在实际行为中的性能。在现实世界的应用中，通常无法完全模拟实际系统的物理特性，因此我们采用包含大规模行为的过程模型。然后，我们调整测量noise和过程noise，直到滤波器能够很好地处理我们的data。这存在着真正的风险；很容易微调KalmanFilter，使其在测试data中完美运行，但在面对稍有不同的data时表现不佳。这可能是设计KalmanFilter最困难的部分，也是为什么它被称为“黑魔法”的原因。

我不喜欢没有解释就实现的书，所以现在我将开发一个球穿过空气移动的物理模型。如果你不感兴趣，可以跳过模拟的实现。

穿过空气移动的球会遇到风阻力。这会给球施加一个力，称为*阻力*，它会改变球的飞行轨迹。在 Giordano 中，这被表示为

$$F_{drag} = -B_2v^2$$

其中 $B_2$ 是通过实验得出的系数，$v$ 是物体的速度。$F_{drag}$ 可以分解为 $x$ 和 $y$ 分量，具体如下

$$\begin{aligned}
F_{drag,x} &= -B_2v v_x\\
F_{drag,y} &= -B_2v v_y
\end{aligned}$$

如果 $m$ 是球的质量，我们可以使用 $F=ma$ 计算加速度，具体如下

$$\begin{aligned} 
a_x &= -\frac{B_2}{m}v v_x\\
a_y &= -\frac{B_2}{m}v v_y
\end{aligned}$$

Giordano提供以下函数来计算$\frac{B_2}{m}$，它考虑了空气密度、棒球的截面和棒球的粗糙度。请注意，这是基于风洞测试和几个简化假设的近似值。它的单位是国际单位制：速度以米/秒为单位，时间以秒为单位。

$$\frac{B_2}{m} = 0.0039 + \frac{0.0058}{1+\exp{[(v-35)/5]}}$$

从真空中球的路径的欧拉离散化开始：

$$\begin{aligned}
x &= v_x \Delta t \\
y &= v_y \Delta t \\
v_x &= v_x \\
v_y &= v_y - 9.8 \Delta t
\end{aligned}
$$

我们可以通过在速度更新方程中加入 $accel * \Delta t$ 来将这个力（加速度）合并到我们的方程中。我们应该减去这个分量，因为阻力会减小速度。代码非常简单，我们只需要将力分解为 $x$ 和 $y$ 分量即可。

我不会进一步详述这个问题，因为计算物理学超出了本书的范围。请认识到，更高保真度的模拟需要纳入高度、温度、球旋转等几个因素。如果您感兴趣，Alan Nathan 的前述工作涵盖了这一点。我在这里的意图是将一些真实世界的行为融入我们的模拟中，以测试我们的KalmanFilter使用的简单预测模型如何对这种行为做出反应。您的过程模型将永远无法精确地模拟发生在世界上的事情，而设计良好的KalmanFilter的一个重要因素就是仔细地测试它在真实世界data中的表现。

以下代码计算海平面上带有风的棒球运动行为。我将绘制无风的初始击球以及10英里每小时的顺风情况下的击球。棒球统计通常采用美国单位制，我们也将在此遵循此惯例。（http://en.wikipedia.org/wiki/United_States_customary_units）。请注意，110英里每小时的速度是击出本垒打的棒球的典型离场速度。

```python
from math import sqrt, exp

def mph_to_mps(x):
    return x * .447

def drag_force(velocity):
    """ 返回指定速度下由空气阻力产生的棒球力。单位为SI"""

    return velocity * (0.0039 + 0.0058 / 
            (1. + exp((velocity-35.)/5.)))

v = mph_to_mps(110.)
x, y = 0., 1.
dt = .1
theta = radians(35)

In [ ]:
def solve(x, y, vel, v_wind, launch_angle):
    xs = []
    ys = []
    v_x = vel*cos(launch_angle)
    v_y = vel*sin(launch_angle)
    while y >= 0:
        # x和y轴的欧拉方程
        x += v_x*dt
        y += v_y*dt

        # 空气阻力力的作用力
        velocity = sqrt((v_x-v_wind)**2 + v_y**2)    
        F = drag_force(velocity)

        # vx和vy的欧拉方程
        v_x = v_x - F*(v_x-v_wind)*dt
        v_y = v_y - 9.8*dt - F*v_y*dt
        
        xs.append(x)
        ys.append(y)
    
    return xs, ys
        
x, y = solve(x=0, y=1, vel=v, v_wind=0, launch_angle=theta)
p1 = plt.scatter(x, y, color='blue', label='无风')

wind = mph_to_mps(10)
x, y = solve(x=0, y=1, vel=v, v_wind=wind, launch_angle=theta)
p2 = plt.scatter(x, y, color='green', marker="v", 
                 label='10mph的风')
plt.legend(scatterpoints=1);

我们可以很容易地看到真空和空气中弹道的差别。我在上面的真空中的球部分使用了相同的初始速度和发射角度。我们计算出，真空中的球将行进超过240米（近800英尺）。在空气中，距离仅超过120米，或大约400英尺。400英尺是一个好的全垒打球的现实距离，因此我们可以相信我们的模拟是相当准确的。

不再多说，我们将创建一个球模拟，使用上述数学来创建更真实的球轨迹。需要注意的是，阻力的非线性行为意味着在任何时间点上，球的位置都没有解析解，因此我们需要逐步计算位置。我使用欧拉方法来传播解决方案；更准确的技术如龙格-库塔方法留给读者作为练习。对于我们正在做的事情来说，这种适度的复杂性是不必要的，因为在我们使用的时间步长下，技术之间的精度差异将很小。

In [ ]:
class BaseballPath:
    def __init__(self, x0, y0, launch_angle_deg, velocity_ms, 
                 noise=(1.0, 1.0)): 
        """ 创建一个二维棒球路径对象  
           (x = 地平面上距离起点的距离， 
            y = 地面上方高度)
        
        x0,y0            初始位置
        launch_angle_deg 球相对于地平面的发射角度
        velocity_ms      球的速度，单位是米/秒
        noise            每个位置添加的noise量 (x, y)
        """
        
        omega = radians(launch_angle_deg)
        self.v_x = velocity_ms * cos(omega)
        self.v_y = velocity_ms * sin(omega)

        self.x = x0
        self.y = y0
        self.noise = noise

```python
def drag_force(self, velocity):
        """ 返回在指定速度下，棒球由于空气阻力所受的力。单位是国际单位制
        """
        B_m = 0.0039 + 0.0058 / (1. + exp((velocity-35.)/5.))
        return B_m * velocity


    def update(self, dt, vel_wind=0.):
        """ 根据指定的时间步长和风速计算球的位置。返回(x, y)位置元组。
        """

        # x和y的欧拉方程
        self.x += self.v_x*dt
        self.y += self.v_y*dt

        # 空气阻力力
        v_x_wind = self.v_x - vel_wind
        v = sqrt(v_x_wind**2 + self.v_y**2)
        F = self.滤波器(v)

        # 速度的欧拉方程
        self.v_x = self.v_x - F*v_x_wind*dt
        self.v_y = self.v_y - 9.81*dt - F*self.v_y*dt

        return (self.x + randn()*self.noise[0], 
                self.y + randn()*self.noise[1])

现在我们可以对这个模型生成的测量结果进行卡尔曼滤波器测试。

In [ ]:
x, y = 0, 1.

theta = 35. # 发射角度
v0 = 50.
dt = 1/10.   # 时间步长
g = np.array([[-9.8]])

plt.figure()
ball = BaseballPath(x0=x, y0=y, launch_angle_deg=theta,
                    velocity_ms=v0, noise=[.3,.3])
f1 = ball_kf(x, y, theta, v0, dt, r=1.)
f2 = ball_kf(x, y, theta, v0, dt, r=10.)
t = 0
xs, ys = [], []
xs2, ys2 = [], []

while f1.x[2] > 0:
    t += dt
    x, y = ball.update(dt)
    z = np.array([[x, y]]).T

    f1.update(z)
    f2.update(z)
    xs.append(f1.x[0])
    ys.append(f1.x[2])
    xs2.append(f2.x[0])
    ys2.append(f2.x[2])    
    f1.predict(u=g) 
    f2.predict(u=g)
    
    p1 = plt.scatter(x, y, color='r', marker='.', s=75, alpha=0.5)

p2, = plt.plot(xs, ys, lw=2)
p3, = plt.plot(xs2, ys2, lw=4)
plt.legend([p1, p2, p3], 
           ['测量', '滤波器(R=0.5)', '滤波器(R=10)'],
           loc='best', scatterpoints=1);

我已经绘制了两种不同Kalman滤波器设置的输出。测量值表示为绿色圆圈，R=0.5的Kalman滤波器表示为细绿线，R=10的Kalman滤波器表示为厚蓝线。这些R值的选择仅用于显示测量噪声对输出的影响，它们并不意味着正确的设计。

我们可以看到两个滤波器都表现不佳。起初，两者都很好地跟踪测量值，但随着时间的推移，它们都发散了。这是因为空气阻力的状态模型是非线性的，而Kalman滤波器则假定它是线性的。如果你回想一下我们在g-h滤波器一章中对非线性的讨论，我们展示了为什么g-h滤波器总是落后于系统的加速度。我们在这里看到了同样的情况——加速度是负的，因此Kalman滤波器始终会超过球的位置。只要加速度继续，滤波器就无法追赶，因此滤波器将继续发散。

我们能做些什么来改善这个问题？最好的方法是使用非线性的卡尔曼滤波器进行滤波，在随后的章节中我们将会做到这一点。然而，这个问题也有一个被称为“工程”解决方案的方法。我们的卡尔曼滤波器假设球在真空中，因此不存在过程噪声。然而，由于球在空气中，大气对球施加了一个力。我们可以把这个力看作是过程噪声。这不是一个特别严谨的想法；首先，这个力远非高斯分布。其次，我们可以计算这个力，所以举手投降说“它是随机的”并不会导致最优解。但是，让我们看看如果我们遵循这条思路会发生什么。

以下代码实现了与之前相同的卡尔曼滤波器，但是加入了非零的过程噪声。我绘制了两个示例，一个是 `Q=.1`，另一个是 `Q=0.01`。

In [ ]:
def plot_ball_with_q(q, r=1., noise=0.3):
    x, y = 0., 1.
    theta = 35. # 发射角度
    v0 = 50.
    dt = 1/10.   # 时间步长
    g = np.array([[-9.8]])

    ball = BaseballPath(x0=x, 
                        y0=y, 
                        launch_angle_deg=theta, 
                        velocity_ms=v0, 
                        noise=[noise,noise])
    f1 = ball_kf(x, y, theta, v0, dt, r=r, q=q)
    t = 0
    xs, ys = [], []

    while f1.x[2] > 0: # 当球未落地时
        t += dt
        x, y = ball.update(dt)
        z = np.array([[x, y]]).T

        f1.update(z)
        xs.append(f1.x[0])
        ys.append(f1.x[2]) 
        f1.predict(u=g) 

        p1 = plt.scatter(x, y, c='r', marker='.', s=75, alpha=0.5)

    p2, = plt.plot(xs, ys, lw=2, color='b')
    plt.legend([p1, p2], ['测量值', 'KalmanFilter'])
    plt.show()

plot_ball_with_q(0.01)
plot_ball_with_q(0.1)

第二个滤波器对测量值跟踪得非常好。看起来有点滞后，但非常小。

这是一个好的技术吗？通常不是，但这取决于情况。在这里，球的力非线性相当恒定和规律。假设我们试图跟踪一辆汽车 - 加速度会随着汽车变速和转弯而变化。当我们使过程噪声高于系统中的实际噪声时，滤波器将选择更加重视测量值。如果您的测量值中噪声不多，则此方法可能适用于您。但是，请考虑下面的这个绘图，其中我增加了测量中的噪声。

In [ ]:
plot_ball_with_q(0.01, r=3, noise=3.)
plot_ball_with_q(0.1, r=3, noise=3.)

这个输出非常糟糕。滤波器别无选择，只能给测量值比过程（预测）步骤更高的权重，但当测量值存在噪声时，滤波器输出只会跟踪噪声。这个线性卡尔曼滤波器固有的限制是导致非线性版本滤波器开发的原因。

话虽如此，使用过程噪声来处理系统中的小非线性是完全可能的。这是卡尔曼滤波器的"黑魔法"的一部分。我们对传感器和系统的模型永远不是完美的。传感器是非高斯的，我们的过程模型也永远不是完美的。您可以通过将测量误差和过程误差设置为比它们的理论正确值高来掩盖其中一些问题，但这样做的代价是非最优解。当然，非最优解总比卡尔曼滤波器发散好。然而，正如我们在上面的图表中所看到的，滤波器的输出很容易变得很糟糕。经常进行许多模拟和测试，并最终得到在这些条件下表现非常出色的滤波器，然后，当您在实际数据上使用滤波器时，条件略有不同，滤波器的性能就会变得很差。

暂时我们会将这个问题搁置一下，因为我们在这个例子中明显地错误地应用了卡尔曼滤波器。在接下来的章节中，我们将回顾这个问题，看看使用各种非线性技术的效果。在某些领域，你可以使用线性卡尔曼滤波器解决非线性问题，但通常你需要使用本书中学到的一种或多种技术。

## 参考文献

[1] Bar-Shalom, Yaakov, et al. *Estimation with Applications to Tracking and Navigation.* John Wiley & Sons, 2001.